<a href="https://colab.research.google.com/github/LikithaC17/GramaSetu-AI/blob/main/GramaSetu_AI_Dataset_Cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Phase 3 – Dataset Improvement (Version 2)


Step 1: Upload and load your CSV in Google Colab

In [3]:

import pandas as pd
from google.colab import files
from io import BytesIO

uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(BytesIO(uploaded[file_name]))

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


Saving gramasetu_rural_commerce (augmented) (1).csv to gramasetu_rural_commerce (augmented) (1) (1).csv
Dataset loaded successfully!
Dataset shape: (19929, 5)

Column names:
['original_prompt', 'original_completion', 'enhanced_prompt', 'enhanced_completion', 'reasoning_trace']

First 5 rows:


,original_prompt,original_completion,enhanced_prompt,enhanced_completion,reasoning_trace
0,Graph the region described by the system of in...,Region enclosed by the triangle with the verti...,Graph the region described by the system of in...,To graph the region described by the system of...,Here's a thinking process that leads to the su...
1,"You are Digital Krishna, a Krishna-inspired AI...",To conduct a meta-review of your governance mo...,"You are Digital Krishna, a Krishna-inspired AI...",Governance delays often accumulate where autho...,Thinking Process:\n\n1. **Analyze the Request...
2,"[{""role"": ""system"", ""content"": ""You are Hazina...",Perfect! I've successfully reconciled your MTN...,"[{""role"": ""system"", ""content"": ""You are Hazina...",Based on the MTN MoMo reconciliation report fo...,"Okay, let me approach this systematically. The..."
3,We've got a mixed load of field-packed leafy g...,Keep the current workflow only if you can reli...,# Context: Leafy Greens Transport and Staging\...,### Workflow Trade-Off Analysis\n\n**1. Qualit...,"Alright, let me break down what we're dealing ..."
4,Give a numbered list of instructions to set up...,"1. Figure out your reference stance, or how fa...",Provide a numbered list of step-by-step instru...,"1. Lay the snowboard flat on a stable, level s...",NaN



Missing values:
original_prompt          909
original_completion      865
enhanced_prompt            0
enhanced_completion        0
reasoning_trace        10056
dtype: int64

Duplicate rows: 0


Step 2 – Keyword matching results

In [4]:

import re

categories = {
    "Group Buying": r"group buying|bulk purchase|collective purchase|bulk order",
    "Kirana Inventory": r"kirana|inventory|reorder point|safety stock|stock management",
    "Transportation": r"transportation|logistics|freight|vehicle capacity|delivery cost",
    "Farmer Market Access": r"farmer|farmers|agricultural market|mandi|crop sales",
    "Artisan Market Access": r"artisan|handicraft|handloom|craft products",
    "Women's SHG": r"\bSHG\b|self.help group|women.s group",
    "Youth Employment": r"rural youth|youth employment|job placement|skill training"
}

text_data = (
    df["enhanced_prompt"].astype(str) + " " +
    df["enhanced_completion"].astype(str)
).str.lower()

category_counts = {}

for category, pattern in categories.items():
    count = text_data.str.contains(pattern, regex=True).sum()
    category_counts[category] = count

print("GRAMASETU CATEGORY KEYWORD COUNTS")
print("---------------------------------")

for category, count in category_counts.items():
    print(f"{category}: {count}")

print("\nTotal rows:", len(df))


GRAMASETU CATEGORY KEYWORD COUNTS
---------------------------------
Group Buying: 659
Kirana Inventory: 1207
Transportation: 1569
Farmer Market Access: 2953
Artisan Market Access: 185
Women's SHG: 335
Youth Employment: 308

Total rows: 19929


Step 3 – Inspect actual matching examples

In [5]:

import pandas as pd

sample_records = []

for category, pattern in categories.items():
    matched = df[
        text_data.str.contains(pattern, regex=True, na=False)
    ]

    sample = matched.sample(
        n=min(5, len(matched)),
        random_state=42
    )

    for index, row in sample.iterrows():
        sample_records.append({
            "Category": category,
            "Row_Index": index,
            "Enhanced_Prompt": row["enhanced_prompt"],
            "Enhanced_Completion": row["enhanced_completion"]
        })

sample_df = pd.DataFrame(sample_records)

pd.set_option("display.max_colwidth", 250)

display(sample_df)


,Category,Row_Index,Enhanced_Prompt,Enhanced_Completion
0,Group Buying,19059,"Leveraging policy analysis, design a bulk purchase of files. Cost ₹1,000. Organization. Retrieval. Value: High. Organize the labeling. Draft the filing system. Plan the bulk purchase of dividers.","# Rural Commerce Filing System Plan (Budget: ₹1,000)\n\n## 1. Executive Summary\nThis plan designs a high-value, physical records management system tailored for rural micro-enterprises (e.g., Farmer Producer Organizations, Self-Help Groups, or Vi..."
1,Group Buying,14433,"Revisiting the horticulturist role, I emphasize ecological balance. Create a prompt scenario in a Karnataka village where pollinator-friendly farmers and an SHG making honey need to bulk-buy bee hives and native seeds. Data: 20 farmers need 40 hi...","### 1. The Prompt Scenario\n\n**Role:** You are a Rural Supply Chain and Ecological Agriculture Coordinator operating in Karnataka, India.\n\n**Context:** A collective of 20 pollinator-friendly farmers and 2 Women's Self-Help Groups (SHGs) focuse..."
2,Group Buying,4973,Draft an ethical leadership guideline for a corporate consortium investing in rural Karnataka through the GramaSetu platform. The guideline must navigate the trade-offs between maximizing efficiency in group buying and ensuring inclusive particip...,"# Ethical Leadership Guideline: GramaSetu Rural Investment Consortium\n**Region:** Rural Karnataka, India \n**Focus:** Agricultural Input Group Buying (Fertilizer & Seeds) \n**Version:** 1.0 \n\n## 1. Contextual Analysis: Rural Commerce Challe..."
3,Group Buying,11327,"As a microfinance policy economist, evaluate the impact of a government-subsidized bulk buying scheme for seeds in Odisha. The subsidy covers 30% of the cost for groups larger than 50 farmers. A village has 45 interested farmers. If they recruit ...",# Economic Impact Evaluation: Seed Subsidy Bulk Buying Scheme (Odisha)\n\n**Role:** Microfinance Policy Economist \n**Subject:** Cost-Benefit Analysis of Group Expansion for Subsidy Eligibility \n\n## 1. Economic Impact Analysis & Net Benefit C...
4,Group Buying,3201,"As an organizational transformation lead, design a process reengineering workflow for a cluster of 15 Kirana stores in a semi-arid village near Tumakuru. The objective is to implement a collective inventory management system that triggers automat...",# Organizational Transformation Plan: Collective Inventory Management for Tumakuru Kirana Cluster\n\n## 1. Executive Summary\nThis document outlines a process reengineering workflow for a cluster of 15 Kirana stores in a semi-arid village near Tu...
5,Kirana Inventory,1716,# Task: Generate Full Risk Assessment Matrix\n\n## Context & Priority\nThe Board has requested the **Full Risk Assessment Matrix** (replacing the previously considered Q3 report). This is a high-priority request.\n\n## Deadline\n- **Final Deliver...,"**Subject: URGENT: Full Risk Assessment Matrix – Framework & Data Structure for Board Submission**\n\n**Priority:** High\n**Deadline:** Internal Draft Today 4:00 PM | Final Board Pack Friday COB\n\nPer the Board's request, we are pivoting immedia..."
6,Kirana Inventory,4092,"Scenario: A village Kirana store is struggling with the 'last-mile' delivery of bulk goods. The store buys 100 bags of cement and 50 bags of fertilizer every month. The wholesaler is 30km away. \n\nContext: The store owner pays a transporter ₹5,0...","This scenario highlights a classic ""last-mile"" logistics challenge in rural commerce: underutilized vehicle capacity leading to high per-unit transport costs. By transitioning from individual procurement to a collaborative ""cluster-based"" model, ..."
7,Kirana Inventory,17918,"Calculate the total farm-gate price for Brown grade sesame seeds (beniseed) in Bassa, Kogi, Nigeria, during the 2025 Lean season. Incorporate the following specific data points: base price of 1575.0, logistics cost of 10.0, and a market adjustmen...","**Subject: Farm-Gate Pricing Analysis – Brown Grad

Step 3 – Display sample records (simplified)

In [6]:

for category, pattern in categories.items():
    matched = df[
        text_data.str.contains(pattern, regex=True, na=False)
    ]

    print("\n" + "=" * 70)
    print("CATEGORY:", category)
    print("=" * 70)

    for index, row in matched.head(2).iterrows():
        print("\nRow:", index)
        print("PROMPT:", str(row["enhanced_prompt"])[:500])
        print("COMPLETION:", str(row["enhanced_completion"])[:500])



CATEGORY: Group Buying

Row: 37
PROMPT: As an agroecologist studying biomass, I see the efficiency in collective action. Construct a prompt scenario in a Karnataka village where farmers growing sugarcane and an SHG producing bio-briquettes need to coordinate a bulk purchase of binding agents and processing machinery parts. The prompt must include: 15 cane farmers needing 300kg of binder; 2 SHGs needing 100kg of binder and 5 specific machine gears. Local inventory is insufficient (50kg binder, 0 gears). The AI must be tasked with: 1) C
COMPLETION: # Scenario: Collective Biomass Supply Chain Coordination in Karnataka

**Location:** A village in Karnataka (e.g., Navalgund, Dharwad District).
**Context:** A cluster of 15 sugarcane farmers and 2 Women's Self-Help Groups (SHGs) producing bio-briquettes are attempting to optimize input costs and ensure timely production through collective procurement. 

Below is the constructed prompt scenario, followed by a comprehensive operational plan, p

Step 4 – Improve category filtering

In [7]:

improved_categories = {
    "Group Buying": r"group buying|bulk purchase|collective procurement|aggregated order|shared procurement",
    "Kirana Inventory": r"kirana store|kirana shop|reorder point|safety stock|retail stock management|shop inventory",
    "Transportation": r"rural transportation|shared transport|transport cost|vehicle capacity|delivery route|logistics optimization",
    "Farmer Market Access": r"farmer market access|market linkage|farm gate price|crop sales|selling crops|agricultural buyers|mandi price",
    "Artisan Market Access": r"artisan marketplace|handicraft sales|handloom market|craft marketplace|artisan products",
    "Women's SHG": r"self.help group|women.s SHG|SHG management|SHG financial management|SHG production",
    "Youth Employment": r"rural youth employment|youth job placement|youth skill training|employment among trained|rural employment program"
}

improved_counts = {}

for category, pattern in improved_categories.items():
    count = text_data.str.contains(
        pattern, regex=True, na=False
    ).sum()
    improved_counts[category] = count

print("IMPROVED KEYWORD COUNTS")
print("-" * 40)

for category, count in improved_counts.items():
    print(f"{category}: {count}")


IMPROVED KEYWORD COUNTS
----------------------------------------
Group Buying: 549
Kirana Inventory: 252
Transportation: 491
Farmer Market Access: 122
Artisan Market Access: 0
Women's SHG: 334
Youth Employment: 26


Step 5 – Inspect three examples per category

In [8]:

for category, pattern in improved_categories.items():
    matched = df[
        text_data.str.contains(pattern, regex=True, na=False)
    ]

    print("\n" + "=" * 80)
    print("CATEGORY:", category)
    print("MATCHES:", len(matched))
    print("=" * 80)

    for index, row in matched.head(3).iterrows():
        print(f"\nROW INDEX: {index}")
        print("PROMPT:")
        print(str(row["enhanced_prompt"])[:700])
        print("\nCOMPLETION:")
        print(str(row["enhanced_completion"])[:700])
        print("-" * 60)

print("\nARTISAN KEYWORD CHECK")

artisan_pattern = r"artisan|handicraft|handloom|weaving|pottery|basket making|craft products"

artisan_matches = text_data.str.contains(
    artisan_pattern, regex=True, na=False
)

print("Broader artisan matches:", artisan_matches.sum())



CATEGORY: Group Buying
MATCHES: 549

ROW INDEX: 37
PROMPT:
As an agroecologist studying biomass, I see the efficiency in collective action. Construct a prompt scenario in a Karnataka village where farmers growing sugarcane and an SHG producing bio-briquettes need to coordinate a bulk purchase of binding agents and processing machinery parts. The prompt must include: 15 cane farmers needing 300kg of binder; 2 SHGs needing 100kg of binder and 5 specific machine gears. Local inventory is insufficient (50kg binder, 0 gears). The AI must be tasked with: 1) Calculating the total order; 2) Sourcing from a regional industrial hub 80km away; 3) Organizing a joint transport with a heavy goods vehicle (capacity 5 tons) that is returning empty from a nearby tow

COMPLETION:
# Scenario: Collective Biomass Supply Chain Coordination in Karnataka

**Location:** A village in Karnataka (e.g., Navalgund, Dharwad District).
**Context:** A cluster of 15 sugarcane farmers and 2 Women's Self-Help Groups (SH

In [9]:
artisan_pattern = r"artisan|handicraft|handloom|weaving|pottery|basket making|craft products"

artisan_matches = df[
    text_data.str.contains(artisan_pattern, regex=True, na=False)
]

print("Total artisan candidates:", len(artisan_matches))

for index, row in artisan_matches.head(10).iterrows():
    print("\nROW INDEX:", index)
    print("PROMPT:", row["enhanced_prompt"][:1000])
    print("COMPLETION:", row["enhanced_completion"][:1000])
    print("-" * 80)

Total artisan candidates: 241

ROW INDEX: 222
PROMPT: Analyze how processing shea nuts into value-added products, such as shea butter, contributes to economic development in Northern Uganda. In your response, address the following key areas:

1. **Economic Impact**: Explain how local processing increases income for farmers and communities compared to selling raw nuts.
2. **Employment Opportunities**: Describe the job creation potential within the processing supply chain.
3. **Market Value**: Discuss the difference in market price between raw shea nuts and processed shea butter.
4. **Community Development**: Highlight any broader social or infrastructure benefits derived from this industry in the region.

Ensure the tone is informative and the analysis is specific to the context of Northern Uganda.
COMPLETION: **Value Addition in the Shea Nut Value Chain: Economic Development in Northern Uganda**

In Northern Uganda, particularly across the Acholi and Lango sub-regions, the shea tree (*

Find more direct artisan-market examples

In [10]:
artisan_market_pattern = (
    r"handicraft sales|artisan marketplace|"
    r"sell.*handicraft|market access.*artisan|"
    r"artisan.*market access|craft.*export|"
    r"artisan.*buyer|weaver.*market|"
    r"direct.to.consumer|digital storefront|"
    r"online marketplace.*artisan|"
    r"exporter.*artisan|tourist.*handicraft"
)

artisan_market_matches = df[
    text_data.str.contains(
        artisan_market_pattern,
        regex=True,
        na=False
    )
]

print("Direct artisan-market candidates:", len(artisan_market_matches))

for index, row in artisan_market_matches.head(15).iterrows():
    print("\nROW INDEX:", index)
    print("PROMPT:", row["enhanced_prompt"][:1000])
    print("COMPLETION:", row["enhanced_completion"][:1000])
    print("-" * 80)

Direct artisan-market candidates: 166

ROW INDEX: 36
PROMPT: # Task: Explain Food Security for Farmers

## Objective
Provide a comprehensive explanation of **food security** tailored specifically to what a farmer needs to know about this topic.

## Key Requirements
1. **Definition**: Clearly define "food security" and any related technical terms or acronyms.
2. **Relevance**: Focus on practical implications, strategies, and knowledge essential for agricultural producers.
3. **Context**: Incorporate the specific reference provided below into your explanation.

## Reference Content
The following context must be addressed in your response:
"Example 4405"

## Output Guidelines
- Use clear headers and bullet points for readability.
- Maintain an educational and professional tone suitable for an agricultural audience.
- Ensure the explanation is actionable and directly applicable to farming operations.
COMPLETION: # Food Security: A Strategic Guide for Agricultural Producers

## Defining Foo

Search for more artisan-specific examples

In [11]:
artisan_patterns = {
    "Artisan Direct Sales": r"artisan.{0,100}(sell|selling|sales|buyer|customer)",
    "Handicraft Marketing": r"handicraft.{0,100}(market|sales|buyer|export|platform)",
    "Weaver Market Access": r"weaver.{0,100}(market|buyer|sales|export)",
    "Digital Artisan Marketplace": r"(artisan|craft|handloom).{0,100}(digital marketplace|online platform|e-commerce)",
    "Craft Export": r"(artisan|handicraft|handloom|craft).{0,100}(export|international market)",
    "Artisan Pricing": r"(artisan|handicraft|handloom).{0,100}(pricing|profit margin|selling price)"
}

for category, pattern in artisan_patterns.items():
    matches = df[
        text_data.str.contains(pattern, regex=True, na=False)
    ]

    print(f"\n{category}: {len(matches)} candidates")

    for index, row in matches.head(3).iterrows():
        print("\nROW:", index)
        print("PROMPT:", row["enhanced_prompt"][:700])
        print("COMPLETION:", row["enhanced_completion"][:500])

/tmp/ipykernel_1594/2743812873.py:12: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  text_data.str.contains(pattern, regex=True, na=False)



Artisan Direct Sales: 39 candidates

ROW: 241
PROMPT: You are a financial economist analyzing commercial vs. tourism exchange rates in rural trade. A village near a tourist spot sells handicrafts. Tourists pay in USD/Euro. Locals bulk-buy materials in INR. Exchange rate: Commercial ₹82/$, Tourist (cash) ₹90/$. Artisans sell to tourists at $10/item (₹900). Material cost: ₹400. If they bulk-sell to an exporter instead, price is ₹700/item (paid in INR at commercial rate). Calculate the margin difference. If 50% of sales are to tourists and 50% to exporters, calculate the weighted average margin. Propose a 'Bulk Export' strategy where artisans pool goods to meet exporter volume, sacrificing the tourist premium for volume stability. Quantify the ris
COMPLETION: # Financial Analysis: Rural Handicraft Trade & Exchange Rate Arbitrage

## 1. Executive Summary
As a financial economist, I have analyzed the margin structures facing rural artisans operating in a dual-market environment: direct-to-

Extract the shortlisted records

In [12]:
selected_rows = [241, 8951, 9204, 1980]

for index in selected_rows:
    row = df.loc[index]

    print("\n" + "=" * 100)
    print("ROW INDEX:", index)
    print("\nENHANCED PROMPT:\n")
    print(row["enhanced_prompt"])
    print("\nENHANCED COMPLETION:\n")
    print(row["enhanced_completion"])
    print("\n" + "=" * 100)


ROW INDEX: 241

ENHANCED PROMPT:

You are a financial economist analyzing commercial vs. tourism exchange rates in rural trade. A village near a tourist spot sells handicrafts. Tourists pay in USD/Euro. Locals bulk-buy materials in INR. Exchange rate: Commercial ₹82/$, Tourist (cash) ₹90/$. Artisans sell to tourists at $10/item (₹900). Material cost: ₹400. If they bulk-sell to an exporter instead, price is ₹700/item (paid in INR at commercial rate). Calculate the margin difference. If 50% of sales are to tourists and 50% to exporters, calculate the weighted average margin. Propose a 'Bulk Export' strategy where artisans pool goods to meet exporter volume, sacrificing the tourist premium for volume stability. Quantify the risk reduction.

ENHANCED COMPLETION:

# Financial Analysis: Rural Handicraft Trade & Exchange Rate Arbitrage

## 1. Executive Summary
As a financial economist, I have analyzed the margin structures facing rural artisans operating in a dual-market environment: direct-

Create a correction tracker

In [13]:
correction_tracker = pd.DataFrame([
    {
        "row_index": 241,
        "category": "Artisan Market Access",
        "status": "Needs correction",
        "issue": "Distinguish income range from variance; clarify assumptions and exchange rates"
    },
    {
        "row_index": 8951,
        "category": "Artisan Market Access",
        "status": "Needs correction",
        "issue": "Recommendation contradicts profit calculation; clarify payment timing"
    },
    {
        "row_index": 9204,
        "category": "Artisan Market Access",
        "status": "Approved for now",
        "issue": "Core bulk packaging calculations are correct"
    },
    {
        "row_index": 1980,
        "category": "Artisan Market Access / SHG / Youth Employment",
        "status": "Needs correction",
        "issue": "Correct bulk threshold and separate one-time savings from monthly salary"
    }
])

display(correction_tracker)

,row_index,category,status,issue
0,241,Artisan Market Access,Needs correction,Distinguish income range from variance; clarify assumptions and exchange rates
1,8951,Artisan Market Access,Needs correction,Recommendation contradicts profit calculation; clarify payment timing
2,9204,Artisan Market Access,Approved for now,Core bulk packaging calculations are correct
3,1980,Artisan Market Access / SHG / Youth Employment,Needs correction,Correct bulk threshold and separate one-time savings from monthly salary


Version 2 dataset.

Step 1: Create corrected examples in Google Colab

In [14]:

corrections = [
    {
        "row_index": 241,
        "category": "Artisan Market Access",
        "correction": """Clearly distinguish gross revenue, expenses, net income, and income ranges. Do not call a range a variance. Use only the exchange rates and costs supplied in the prompt. Clearly label any additional figures as assumptions or suggestions, not facts. Show calculations step by step."""
    },
    {
        "row_index": 8951,
        "category": "Artisan Market Access",
        "correction": """Compare both selling options using the calculated net profits. Option A: 50 baskets sold to the boutique, revenue ₹3,40,000, costs ₹2,03,000, profit ₹1,37,000. Option B: 40 baskets sold to the boutique and 10 locally, revenue ₹3,45,000, costs ₹2,03,000, profit ₹1,42,000. Option B earns ₹5,000 more. Do not recommend Option A based on these figures. Discuss payment timing and holding costs only as unknown factors requiring confirmation."""
    },
    {
        "row_index": 1980,
        "category": "Artisan Market Access / SHG / Youth Employment",
        "correction": """The bulk price of ₹110 per kg applies only when purchasing more than 20 kg. A 20 kg purchase therefore does not qualify. At 21 kg, the purchase costs ₹2,310, compared with ₹3,000 for 20 kg at ₹150 per kg. The difference is ₹690, but the bulk option includes 1 kg extra inventory. Calculate the youth's monthly salary of ₹3,000 plus 5% commission on ₹10,000 sales (₹500), giving ₹3,500 monthly. Clearly distinguish one-time purchasing savings from recurring monthly employment costs."""
    }
]

corrections_df = pd.DataFrame(corrections)
display(corrections_df)


,row_index,category,correction
0,241,Artisan Market Access,"Clearly distinguish gross revenue, expenses, net income, and income ranges. Do not call a range a variance. Use only the exchange rates and costs supplied in the prompt. Clearly label any additional figures as assumptions or suggestions, not fact..."
1,8951,Artisan Market Access,"Compare both selling options using the calculated net profits. Option A: 50 baskets sold to the boutique, revenue ₹3,40,000, costs ₹2,03,000, profit ₹1,37,000. Option B: 40 baskets sold to the boutique and 10 locally, revenue ₹3,45,000, costs ₹2,..."
2,1980,Artisan Market Access / SHG / Youth Employment,"The bulk price of ₹110 per kg applies only when purchasing more than 20 kg. A 20 kg purchase therefore does not qualify. At 21 kg, the purchase costs ₹2,310, compared with ₹3,000 for 20 kg at ₹150 per kg. The difference is ₹690, but the bulk opti..."


Step 2: Retrieve the original prompts and completions

In [15]:

rows_to_correct = [241, 8951, 1980]

for idx in rows_to_correct:
    print(f"\n{'='*80}")
    print(f"DATASET ROW: {idx}")
    print(f"{'='*80}")
    print("\nORIGINAL PROMPT:\n")
    print(df.loc[idx, "enhanced_prompt"])
    print("\nORIGINAL COMPLETION:\n")
    print(df.loc[idx, "enhanced_completion"])



DATASET ROW: 241

ORIGINAL PROMPT:

You are a financial economist analyzing commercial vs. tourism exchange rates in rural trade. A village near a tourist spot sells handicrafts. Tourists pay in USD/Euro. Locals bulk-buy materials in INR. Exchange rate: Commercial ₹82/$, Tourist (cash) ₹90/$. Artisans sell to tourists at $10/item (₹900). Material cost: ₹400. If they bulk-sell to an exporter instead, price is ₹700/item (paid in INR at commercial rate). Calculate the margin difference. If 50% of sales are to tourists and 50% to exporters, calculate the weighted average margin. Propose a 'Bulk Export' strategy where artisans pool goods to meet exporter volume, sacrificing the tourist premium for volume stability. Quantify the risk reduction.

ORIGINAL COMPLETION:

# Financial Analysis: Rural Handicraft Trade & Exchange Rate Arbitrage

## 1. Executive Summary
As a financial economist, I have analyzed the margin structures facing rural artisans operating in a dual-market environment: direc

Step 3: Create corrected completions

In [16]:

corrected_completions = [
    {
        "row_index": 241,
        "category": "Artisan Market Access",
        "corrected_completion": """# Financial Analysis: Rural Handicraft Trade

## 1. Unit Economics

Tourist channel:
- Selling price: $10 per item
- Tourist cash exchange rate: ₹90/$
- Revenue: $10 × ₹90 = ₹900
- Material cost: ₹400
- Gross profit per item: ₹900 - ₹400 = ₹500
- Gross margin percentage: ₹500 / ₹900 × 100 = 55.56%

Exporter channel:
- Selling price: ₹700 per item
- Material cost: ₹400
- Gross profit per item: ₹700 - ₹400 = ₹300
- Gross margin percentage: ₹300 / ₹700 × 100 = 42.86%

The tourist channel earns ₹200 more gross profit per item than the exporter channel.

The commercial exchange rate of ₹82/$ is not used in the tourist cash calculation because the prompt specifies the tourist cash rate of ₹90/$. It may be relevant for formal currency conversion, but no additional conversion is required for the stated exporter price of ₹700.

## 2. Weighted Average Gross Profit

Assume 100 items are sold, with 50 to tourists and 50 to exporters.

Tourist profit = 50 × ₹500 = ₹25,000.
Exporter profit = 50 × ₹300 = ₹15,000.
Total gross profit = ₹40,000.

Weighted average gross profit = ₹40,000 / 100 = ₹400 per item.

## 3. Bulk Export Strategy

Artisans can form a cooperative, combine production, standardize quality, and negotiate a regular exporter contract. They can retain some tourist sales while using exporter orders to improve demand stability.

An illustrative strategy is to sell 80% to exporters and 20% to tourists.

Important: The proposed reduction in material cost from ₹400 to ₹350 is an assumption, not a figure supplied in the prompt. It must be verified through supplier quotations before being used in a real business decision.

If the material cost remains ₹400:
- Exporter profit per item = ₹700 - ₹400 = ₹300.
- Tourist profit per item = ₹900 - ₹400 = ₹500.
- Weighted average profit = (0.80 × ₹300) + (0.20 × ₹500) = ₹340 per item.

If cooperative bulk purchasing genuinely reduces material cost to ₹350:
- Exporter profit per item = ₹350.
- Tourist profit per item = ₹550.
- Weighted average profit = (0.80 × ₹350) + (0.20 × ₹550) = ₹390 per item.

The second calculation is conditional on achieving the assumed ₹350 material cost.

## 4. Illustrative Income Stability Analysis

The prompt does not provide seasonal demand data. For illustration only, assume:
- Baseline production: 100 items per month.
- Tourist demand fluctuates by ±50%.
- Exporter demand fluctuates by ±10%.

Under the current 50/50 mix, using the stated ₹500 and ₹300 profits:
High-season gross profit = (50 × 1.5 × ₹500) + (50 × 1.1 × ₹300) = ₹54,000.
Low-season gross profit = (50 × 0.5 × ₹500) + (50 × 0.9 × ₹300) = ₹26,000.
Income range = ₹54,000 - ₹26,000 = ₹28,000.

Under the proposed 80% exporter and 20% tourist mix, assuming material cost actually falls to ₹350:
High-season gross profit = (20 × 1.5 × ₹550) + (80 × 1.1 × ₹350) = ₹47,300.
Low-season gross profit = (20 × 0.5 × ₹550) + (80 × 0.9 × ₹350) = ₹30,700.
Income range = ₹47,300 - ₹30,700 = ₹16,600.

Reduction in income range = (₹28,000 - ₹16,600) / ₹28,000 × 100 = 40.71%.

The minimum illustrative monthly gross profit increases from ₹26,000 to ₹30,700, approximately 18.08%.

These figures demonstrate a reduction in the modeled high-to-low income range, not a statistically calculated variance. The seasonal fluctuations and material cost reduction are assumptions that require real sales and supplier data for validation.

## 5. Recommendations

1. Form an artisan cooperative to aggregate production.
2. Negotiate written exporter agreements specifying price, quantity, quality, and payment dates.
3. Obtain supplier quotations before assuming bulk material savings.
4. Maintain a balance between tourist sales and exporter contracts.
5. Track monthly sales, payment delays, costs, and seasonal demand.
6. Use digital payments and maintain transparent financial records.

## Conclusion

Tourist sales offer higher gross profit per item, while exporter contracts may provide more predictable demand. A cooperative can improve bargaining power and reduce dependence on tourist demand. The strategy should be evaluated using verified costs, actual orders, and payment terms rather than treating illustrative assumptions as guaranteed outcomes."""
    },
    {
        "row_index": 8951,
        "category": "Artisan Market Access",
        "corrected_completion": """# Profitability Analysis for the Weaver Cluster

## 1. Production and Transport Costs

Number of weavers: 10.
Total sarees: 50.
Production cost per saree: ₹4,000.

Total production cost = 50 × ₹4,000 = ₹2,00,000.

The tempo costs ₹3,000 and requires a minimum load of 40 sarees to be cost-effective. Both scenarios send 40 sarees to the boutique, so the minimum load is met.

Total cost in either scenario = ₹2,00,000 + ₹3,000 = ₹2,03,000.

## 2. Scenario A: Sell All 50 Sarees to the Boutique

40 sarees × ₹7,000 = ₹2,80,000.
10 sarees × ₹6,000 = ₹60,000.

Total revenue = ₹3,40,000.
Net profit = ₹3,40,000 - ₹2,03,000 = ₹1,37,000.

Profit per weaver = ₹1,37,000 / 10 = ₹13,700.

## 3. Scenario B: Sell 40 to the Boutique and 10 Locally

40 sarees × ₹7,000 = ₹2,80,000.
10 sarees × ₹6,500 = ₹65,000.

Total revenue = ₹3,45,000.
Net profit = ₹3,45,000 - ₹2,03,000 = ₹1,42,000.

Profit per weaver = ₹1,42,000 / 10 = ₹14,200.

## 4. Comparison

Scenario A:
- Total profit: ₹1,37,000.
- Profit per weaver: ₹13,700.

Scenario B:
- Total profit: ₹1,42,000.
- Profit per weaver: ₹14,200.

Scenario B generates ₹5,000 more total profit, or ₹500 more per weaver.

## 5. Recommendation

Based only on the stated selling prices and costs, Scenario B is financially better because selling the 10 sarees locally at ₹6,500 generates ₹5,000 more revenue than selling them to the boutique at ₹6,000.

However, local sales involve a longer waiting period. The prompt does not specify the exact payment dates, probability of local sales, storage costs, or risk of unsold inventory. Therefore, immediate boutique payment must not be assumed.

The cluster should:
1. Confirm the boutique's order and payment terms for 40 sarees.
2. Check whether local buyers are likely to purchase the remaining 10 sarees.
3. Estimate any additional storage, marketing, or delivery expenses.
4. Compare the benefit of the additional ₹5,000 with the cost and risk of waiting.

If local buyers are reliable and the cluster can manage the waiting period without significant additional expenses, Scenario B is preferable.

If the cluster urgently needs cash or local demand is uncertain, Scenario A may be safer despite its ₹5,000 lower calculated profit.

## Conclusion

Both scenarios meet the tempo's 40-saree minimum load. Scenario B produces ₹1,42,000 profit compared with ₹1,37,000 in Scenario A. The final decision should consider payment timing, local demand, and additional holding costs. The available information supports a conditional recommendation rather than assuming one option is always better."""
    },
    {
        "row_index": 1980,
        "category": "Artisan Market Access / SHG / Youth Employment",
        "corrected_completion": """# Financial Analysis for the Women's SHG

## 1. Thread Purchase Requirement

Number of SHG members: 10.
Thread required per member: 2 kg.

Total requirement = 10 × 2 = 20 kg.

Retail price = ₹150 per kg.
Bulk price = ₹110 per kg, applicable only for purchases above 20 kg.

## 2. Bulk Purchase Calculation

Retail cost for 20 kg:
20 × ₹150 = ₹3,000.

A purchase of exactly 20 kg does not qualify for the bulk rate because the condition is strictly above 20 kg.

To qualify, the SHG can purchase 21 kg:
21 × ₹110 = ₹2,310.

Cash expenditure difference compared with purchasing 20 kg at retail:
₹3,000 - ₹2,310 = ₹690.

However, the bulk purchase includes an additional 1 kg of thread. Therefore, ₹690 is the difference in total cash expenditure between these two purchase options, not a like-for-like saving for the same quantity.

For an equal-quantity comparison of 21 kg:
Retail cost = 21 × ₹150 = ₹3,150.
Bulk cost = 21 × ₹110 = ₹2,310.
Saving = ₹3,150 - ₹2,310 = ₹840.

## 3. Youth Employment Cost

Fixed monthly salary = ₹3,000.
Expected monthly sales = ₹10,000.
Commission = 5% × ₹10,000 = ₹500.

Total monthly employment cost = ₹3,000 + ₹500 = ₹3,500.

## 4. Can Bulk Savings Offset Employment Cost?

Using the 21 kg bulk purchase and comparing its cash expenditure with the 20 kg retail purchase:
Cash expenditure difference = ₹690.
One month's employment cost = ₹3,500.

Remaining amount = ₹3,500 - ₹690 = ₹2,810.

The ₹690 difference is a one-time procurement comparison, while ₹3,500 is a recurring monthly employment expense. They are not expenses from the same time period.

Even if the SHG uses the ₹690 difference toward the first month's employment cost, it covers only ₹690 / ₹3,500 × 100 = 19.71% of that monthly cost.

Therefore, the bulk purchase difference does not fully offset the youth's monthly employment cost.

## 5. Recommendations

1. Confirm with the supplier that the ₹110/kg rate applies to a 21 kg purchase.
2. Check whether the additional 1 kg will be used and whether transport charges reduce the benefit.
3. Track actual monthly Etsy sales and calculate the commission transparently.
4. Evaluate whether online sales generate enough additional profit to justify the ₹3,500 monthly employment cost.
5. Maintain transparent SHG records for salary, commission, procurement, and sales.

## Conclusion

The SHG requires 20 kg of thread. To qualify for the strictly above-20-kg bulk rate, it must purchase at least 21 kg. This costs ₹2,310, compared with ₹3,000 for 20 kg at retail, a ₹690 difference in cash expenditure while receiving an additional kilogram.

The youth's employment costs ₹3,500 per month. The procurement difference does not offset this recurring cost. The SHG should assess the additional revenue generated by online sales before deciding whether the employment arrangement is financially sustainable."""
    }
]

corrected_df = pd.DataFrame(corrected_completions)

display(corrected_df[["row_index", "category"]])
print("Corrected examples created:", len(corrected_df))


,row_index,category
0,241,Artisan Market Access
1,8951,Artisan Market Access
2,1980,Artisan Market Access / SHG / Youth Employment


Corrected examples created: 3


Step 4: Create the separate Version 2 dataset

In [17]:

# Create a separate copy for Version 2
df_v2 = df.copy()

# Verify that all correction rows exist
rows_to_update = corrected_df["row_index"].tolist()
missing_rows = [i for i in rows_to_update if i not in df_v2.index]

if missing_rows:
    raise ValueError(f"Rows not found in dataset: {missing_rows}")

# Update only the three enhanced completions
for _, row in corrected_df.iterrows():
    df_v2.loc[row["row_index"], "enhanced_completion"] = row["corrected_completion"]

# Verify the changes
print("Version 1 shape:", df.shape)
print("Version 2 shape:", df_v2.shape)
print("Corrected rows:", rows_to_update)

for idx in rows_to_update:
    print(f"Row {idx}:",
          "UPDATED" if df_v2.loc[idx, "enhanced_completion"] != df.loc[idx, "enhanced_completion"] else "NOT UPDATED")

# Save Version 2 separately
output_file = "/content/gramasetu_rural_commerce_v2.csv"
df_v2.to_csv(output_file, index=False)

print("\nVersion 2 dataset saved:", output_file)
print("Version 2 dataset size:", df_v2.shape[0], "rows")


Version 1 shape: (19929, 5)
Version 2 shape: (19929, 5)
Corrected rows: [241, 8951, 1980]
Row 241: UPDATED
Row 8951: UPDATED
Row 1980: UPDATED

Version 2 dataset saved: /content/gramasetu_rural_commerce_v2.csv
Version 2 dataset size: 19929 rows


Step 5: Validate Version 2

In [18]:

print("VERSION 2 VALIDATION")
print("-" * 45)

print("Dataset shape:", df_v2.shape)

print("\nMissing values:")
print(df_v2.isnull().sum())

print("\nDuplicate rows:", df_v2.duplicated().sum())

print("\nOriginal prompts unchanged:",
      df_v2["enhanced_prompt"].equals(df["enhanced_prompt"]))

print("\nChanged completion count:",
      (df_v2["enhanced_completion"] != df["enhanced_completion"]).sum())

print("\nCorrected row verification:")
for idx in [241, 8951, 1980]:
    print(f"Row {idx}:",
          df_v2.loc[idx, "enhanced_completion"] ==
          corrected_df.loc[corrected_df["row_index"] == idx, "corrected_completion"].iloc[0])

print("\nValidation completed.")


VERSION 2 VALIDATION
---------------------------------------------
Dataset shape: (19929, 5)

Missing values:
original_prompt          909
original_completion      865
enhanced_prompt            0
enhanced_completion        0
reasoning_trace        10056
dtype: int64

Duplicate rows: 0

Original prompts unchanged: True

Changed completion count: 3

Corrected row verification:
Row 241: True
Row 8951: True
Row 1980: True

Validation completed.


Step 6 – Prepare Version 2 training data

In [19]:

# Prepare Version 2 training data
train_df_v2 = df_v2[[
    "enhanced_prompt",
    "enhanced_completion"
]].copy()

# Rename columns for training
train_df_v2.columns = ["prompt", "completion"]

# Remove any empty records
train_df_v2 = train_df_v2.dropna()

# Remove whitespace-only records
train_df_v2["prompt"] = train_df_v2["prompt"].str.strip()
train_df_v2["completion"] = train_df_v2["completion"].str.strip()

train_df_v2 = train_df_v2[
    (train_df_v2["prompt"] != "") &
    (train_df_v2["completion"] != "")
]

# Save training dataset
train_file = "/content/gramasetu_rural_commerce_v2_training.csv"
train_df_v2.to_csv(train_file, index=False)

print("Version 2 training data prepared!")
print("Training shape:", train_df_v2.shape)
print("Saved file:", train_file)
print("\nSample training record:")
print("PROMPT:", train_df_v2.iloc[0]["prompt"][:300])
print("COMPLETION:", train_df_v2.iloc[0]["completion"][:300])


Version 2 training data prepared!
Training shape: (19929, 2)
Saved file: /content/gramasetu_rural_commerce_v2_training.csv

Sample training record:
PROMPT: Graph the region described by the system of inequalities

$$
\left\{\begin{array}{l}
x+y-1 \leqslant 0 \quad(B C) \\
-x+y-1 \leqslant 0 \quad(A B) \\
y \geqslant-1 \quad(A C)
\end{array}\right.
$$
COMPLETION: To graph the region described by the system of inequalities, we determine the boundary lines, identify the vertices of the resulting shape, and define the shading for the region.

### 1. Analyze the Boundary Lines
We rewrite each inequality in slope-intercept form ($y = mx + b$) to identify the line


Step 7: Check dataset category distribution

In [20]:

import re

category_patterns = {
    "Group Buying": r"group buying|bulk purchase|bulk procurement|collective purchase|farmers.*pool",
    "Kirana Inventory": r"kirana|inventory management|stock level|reorder point|safety stock",
    "Transportation": r"transport cost|vehicle capacity|logistics|transportation|delivery route",
    "Farmer Market Access": r"farmer market|market access|agricultural market|sell.*produce|crop.*buyer",
    "Artisan Market Access": r"artisan|handicraft|handloom|weaver|pottery|craft products",
    "Women's SHG": r"self.help group|women's shg|women shg|shg member|shg",
    "Youth Employment": r"youth employment|rural youth|employment generation|job placement|trained youth"
}

text_data = (
    train_df_v2["prompt"] + " " + train_df_v2["completion"]
).str.lower()

print("CATEGORY DISTRIBUTION")
print("-" * 45)

for category, pattern in category_patterns.items():
    count = text_data.str.contains(pattern, regex=True, na=False).sum()
    print(f"{category}: {count}")

print("\nTotal training examples:", len(train_df_v2))


CATEGORY DISTRIBUTION
---------------------------------------------
Group Buying: 611
Kirana Inventory: 409
Transportation: 1582
Farmer Market Access: 309
Artisan Market Access: 196
Women's SHG: 512
Youth Employment: 320

Total training examples: 19929


Step 8 – Inspect the relevance of the training examples

In [21]:

import random

random.seed(42)

for category, pattern in category_patterns.items():
    matches = train_df_v2[
        text_data.str.contains(pattern, regex=True, na=False)
    ]

    sample = matches.sample(
        n=min(3, len(matches)),
        random_state=42
    )

    print("\n" + "=" * 80)
    print("CATEGORY:", category)
    print("=" * 80)

    for idx, row in sample.iterrows():
        print("\nPROMPT:")
        print(row["prompt"][:700])
        print("\nCOMPLETION:")
        print(row["completion"][:500])
        print("-" * 60)



CATEGORY: Group Buying

PROMPT:
Act as a public sector procurement consultant for rural utilities. A village panchayat in Scotland-style governance (adapted to India) wants to bulk-purchase street light bulbs for 50 poles. Specs: LED, 20W. Market rate: ₹400/bulb. Bulk tender (>40 units): ₹280/bulb. Three local vendors bid. Vendor A: ₹290, 2-day delivery. Vendor B: ₹280, 7-day delivery. Vendor C: ₹300, 1-day delivery + free installation. Installation cost if not free: ₹50/pole. Evaluate the bids using a 'Most Economically Advantageous Tender' (MEAT) framework including time-value of light restoration. Calculate the total lifecycle cost for each option over 1 year. Recommend the winner and draft the award notice logic for Gr

COMPLETION:
# Procurement Consultation Report: Rural Street Lighting Bulk Purchase
**To:** Village Panchayat Standing Committee  
**From:** Public Sector Procurement Consultant  
**Subject:** MEAT Evaluation for 50x LED Street Light Bulbs (20W)  
**Platform:** Gram

Step 9 – Identify unrelated training examples

In [22]:

unrelated_patterns = {
    "Healthcare": r"healthcare|medical access|hospital treatment",
    "Politics": r"adam schiff|political allegation|election campaign",
    "Mathematics": r"system of inequalities|graph the region|calculus|differential equation",
    "Unrelated Events": r"September 2024|Kisan Mela|upcoming event",
    "Foreign Location Mismatch": r"Kenya|Uganda|Scotland-style"
}

review_results = []

for category, pattern in unrelated_patterns.items():
    matches = train_df_v2[
        train_df_v2["prompt"].str.contains(
            pattern, case=False, regex=True, na=False
        ) |
        train_df_v2["completion"].str.contains(
            pattern, case=False, regex=True, na=False
        )
    ]

    for idx, row in matches.iterrows():
        review_results.append({
            "index": idx,
            "flag": category,
            "prompt": row["prompt"][:250],
            "completion": row["completion"][:250]
        })

review_df = pd.DataFrame(review_results)

print("Potentially unrelated records:", len(review_df))
display(review_df.head(20))


Potentially unrelated records: 1180


,index,flag,prompt,completion
0,17,Healthcare,Explain the key reasons why I should update my tax return.,Updating a previously filed tax return—formally done by filing Form 1040-X—is a strategic decision that ensures your financial record with the IRS is both accurate and optimized. There are several critical reasons to consider amending a return:\n...
1,58,Healthcare,"# Role\nYou are an expert analyst in African healthcare systems, specializing in the intersection of traditional medicine and modern biomedical practices.\n\n# Task\nAnalyze the provided text to answer the following core questions:\n1. How have t...","### Adaptation of Traditional African Healing Practices to Modern Healthcare Systems\n\nTraditional African healing practices have evolved from ancient indigenous systems—combining herbal medicine, spiritual beliefs, and communal rituals—to navig..."
2,60,Healthcare,"In addition to the given requirements, outline a robust access control mechanism that ensures patient privacy by restricting unauthorized access to medical records. Furthermore, propose an encryption scheme that secures the sensitive medical data...","This proposal outlines a comprehensive architecture for a Secure Electronic Health Record (EHR) System. It addresses access control, encryption, polyglot database schema design, disaster recovery, and dynamic data modeling, adhering to compliance..."
3,98,Healthcare,"Develop a strategic plan to help a small business navigate the complexities of healthcare regulations, focusing on minimizing financial risks and maximizing tax benefits. The plan must include at least three sections and adhere to the following s...",regulatory compliance and risk mitigation\n\nnavigating healthcare regulations requires a robust framework to avoid penalties. focus on hipaa privacy rules and stark law compliance to prevent fraud allegations. maintain detailed records of all pa...
4,169,Healthcare,"# Objective\nExplain how fiscal and monetary policies aim to influence the key factors that contribute to sustainable long-term economic growth.\n\n# Instructions\n1. **Define Key Concepts**: Briefly define ""fiscal policy,"" ""monetary policy,"" and...","# Strategic Framework for Sustainable Long-Term Economic Growth\n\n## 1. Definition of Key Concepts\n\nTo understand how policy influences long-term economic trajectories, it is necessary to establish precise definitions of the tools and the obje..."
5,210,Healthcare,"# Task: Explain Pesticide Handling and Safety\n\nPlease provide a comprehensive guide on how pesticides should be handled, with a specific focus on **Agricultural safety**. Your response must address the following key areas:\n\n1. **General Pest...","**Safety Protocol Document**\n**Case Identifier:** Example case 18380\n**Year:** 2024\n**Topic Tags:** Agricultural safety, Explain pesticide safety, Pesticide safety equipment\n**Media Reference:** photo\n\n---\n\n### Comprehensive Guide to Pest..."
6,222,Healthcare,"Analyze how processing shea nuts into value-added products, such as shea butter, contributes to economic development in Northern Uganda. In your response, address the following key areas:\n\n1. **Economic Impact**: Explain how local processing in...","**Value Addition in the Shea Nut Value Chain: Economic Development in Northern Uganda**\n\nIn Northern Uganda, particularly across the Acholi and Lango sub-regions, the shea tree (*Vitellaria paradoxa*) represents a critical natural asset. Often ..."
7,417,Healthcare,"Analyze the following passage from 'Long Term Memory: The Foundation of AI Self-Evolution' regarding Human Memory and AI Memory (Section 6.2.3: LTM Utilization). Provide a step-by-step reasoning process that breaks down the text's core arguments,...","### 1. Context & Potential\nCurrent Large Language Models (LLMs) and their derivative agents exhibit sophisticated language comprehension capabilities, enabling high-fidelity interactions that approximate human dialogue. In the domain of mental h..."


Step 1 — Separate clearly unrelated records from records needing review

In [23]:
import re
import pandas as pd

project_patterns = {
    "Group Buying": r"group buying|bulk purchase|bulk procurement|collective purchase|group procurement",
    "Kirana Inventory": r"kirana|inventory management|stock level|reorder point|safety stock",
    "Transportation": r"transport cost|vehicle capacity|logistics|transportation|delivery route",
    "Farmer Market Access": r"farmer market|market access|agricultural market|farmer.{0,30}sell|crop.{0,30}buyer",
    "Artisan Market Access": r"artisan|handicraft|handloom|weaver|pottery|craft products",
    "Women's SHG": r"self.help group|women.{0,20}shg|shg member|shg",
    "Youth Employment": r"youth employment|rural youth|employment generation|job placement|trained youth"
}

def classify_record(row):
    prompt = str(row["prompt"]).lower()
    completion = str(row["completion"]).lower()
    combined = prompt + " " + completion

    matches = [
        category for category, pattern in project_patterns.items()
        if re.search(pattern, combined)
    ]

    if matches:
        return "Needs relevance review", ", ".join(matches)

    return "No project keyword found", "None"

review_df[["review_status", "possible_categories"]] = review_df.apply(
    lambda row: pd.Series(classify_record(row)), axis=1
)

print(review_df["review_status"].value_counts())
display(review_df.head(30))

review_status
No project keyword found    1133
Needs relevance review        47
Name: count, dtype: int64


,index,flag,prompt,completion,review_status,possible_categories
0,17,Healthcare,Explain the key reasons why I should update my tax return.,Updating a previously filed tax return—formally done by filing Form 1040-X—is a strategic decision that ensures your financial record with the IRS is both accurate and optimized. There are several critical reasons to consider amending a return:\n...,No project keyword found,None
1,58,Healthcare,"# Role\nYou are an expert analyst in African healthcare systems, specializing in the intersection of traditional medicine and modern biomedical practices.\n\n# Task\nAnalyze the provided text to answer the following core questions:\n1. How have t...","### Adaptation of Traditional African Healing Practices to Modern Healthcare Systems\n\nTraditional African healing practices have evolved from ancient indigenous systems—combining herbal medicine, spiritual beliefs, and communal rituals—to navig...",No project keyword found,None
2,60,Healthcare,"In addition to the given requirements, outline a robust access control mechanism that ensures patient privacy by restricting unauthorized access to medical records. Furthermore, propose an encryption scheme that secures the sensitive medical data...","This proposal outlines a comprehensive architecture for a Secure Electronic Health Record (EHR) System. It addresses access control, encryption, polyglot database schema design, disaster recovery, and dynamic data modeling, adhering to compliance...",No project keyword found,None
3,98,Healthcare,"Develop a strategic plan to help a small business navigate the complexities of healthcare regulations, focusing on minimizing financial risks and maximizing tax benefits. The plan must include at least three sections and adhere to the following s...",regulatory compliance and risk mitigation\n\nnavigating healthcare regulations requires a robust framework to avoid penalties. focus on hipaa privacy rules and stark law compliance to prevent fraud allegations. maintain detailed records of all pa...,No project keyword found,None
4,169,Healthcare,"# Objective\nExplain how fiscal and monetary policies aim to influence the key factors that contribute to sustainable long-term economic growth.\n\n# Instructions\n1. **Define Key Concepts**: Briefly define ""fiscal policy,"" ""monetary policy,"" and...","# Strategic Framework for Sustainable Long-Term Economic Growth\n\n## 1. Definition of Key Concepts\n\nTo understand how policy influences long-term economic trajectories, it is necessary to establish precise definitions of the tools and the obje...",No project keyword found,None
5,210,Healthcare,"# Task: Explain Pesticide Handling and Safety\n\nPlease provide a comprehensive guide on how pesticides should be handled, with a specific focus on **Agricultural safety**. Your response must address the following key areas:\n\n1. **General Pest...","**Safety Protocol Document**\n**Case Identifier:** Example case 18380\n**Year:** 2024\n**Topic Tags:** Agricultural safety, Explain pesticide safety, Pesticide safety equipment\n**Media Reference:** photo\n\n---\n\n### Comprehensive Guide to Pest...",No project keyword found,None
6,222,Healthcare,"Analyze how processing shea nuts into value-added products, such as shea butter, contributes to economic development in Northern Uganda. In your response, address the following key areas:\n\n1. **Economic Impact**: Explain how local processing in...","**Value Addition in the Shea Nut Value Chain: Economic Development in Northern Uganda**\n\nIn Northern Uganda, particularly across the Acholi and Lango sub-regions, the shea tree (*Vitellaria paradoxa*) represents a critical natural asset. Often ...",No project keyword found,None
7,417,Healthcare,"Analyze the following passage from 'Long Term Memory: The Foundation of AI Self-Evolution' regarding Human Memory and AI Memory (Section 6.2.3: LTM Utilization). Provide a step-by-step reasoning process that breaks down the text's core arguments,...","### 1. 

Step 2 — Inspect the 47 candidate records

In [24]:
display(
    review_df[review_df["review_status"] == "Needs relevance review"][
        ["index", "prompt", "completion", "possible_categories"]
    ].to_string(index=False)
)

' index                                                                                                                                                                                                                                                            prompt                                                                                                                                                                                                                                                         completion                                         possible_categories\n  6552                                                                          Leveraging urban horticulture, design a bulk purchase of gloves. Cost ₹2,000. Safety. Hygiene. Value: High. Organize the sizing. Draft the washing rule. Plan the bulk purchase of soap.      # Urban Horticulture Procurement Plan: Gloves & Hygiene\\n\\n## 1. Executive Summary\\nThis plan outlines a high-value bulk purchase strategy fo

Next step — Create a manual review list

In [25]:
review_labels = {
    6552: "relevant",
    6861: "mismatch",
    9600: "needs_review",
    9863: "mismatch",
    10200: "mismatch",
    12989: "mismatch",
    16236: "mismatch",
    200: "mismatch",
    2145: "mismatch",
    3088: "needs_review",
    6855: "mismatch",
    8363: "mismatch",
    10575: "mismatch",
    12285: "mismatch",
    14607: "mismatch",
    14711: "needs_review",
    17199: "mismatch",
    252: "relevant",
    836: "mismatch",
    1209: "mismatch",
    2204: "needs_review",
    3220: "relevant",
    3364: "mismatch",
    4990: "mismatch",
    5897: "mismatch",
    6342: "needs_review",
    6463: "mismatch",
    6473: "mismatch",
    6961: "mismatch",
    7665: "mismatch",
    9225: "needs_review",
    9545: "mismatch",
    9753: "mismatch",
    10082: "relevant",
    10511: "needs_review",
    11387: "needs_review",
    11641: "mismatch",
    11927: "mismatch",
    12925: "needs_review",
    14974: "mismatch",
    15292: "mismatch",
    15513: "needs_review",
    17314: "mismatch",
    17620: "mismatch",
    17738: "mismatch",
    17847: "needs_review"
}

print("Records classified:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

Records classified: 46
mismatch        31
needs_review    11
relevant         4
Name: count, dtype: int64


Next step — Save the review classifications

In [26]:
review_df["manual_classification"] = review_df["index"].map(review_labels)

review_df.to_csv(
    "/content/gramasetu_v2_reviewed_candidates.csv",
    index=False
)

print("Review file saved successfully")
print(review_df["manual_classification"].value_counts())

Review file saved successfully
manual_classification
mismatch        32
needs_review    11
relevant         4
Name: count, dtype: int64


Step 1 — Check duplicate records

In [27]:
print("Total review entries:", len(review_df))
print("Unique dataset rows:", review_df["index"].nunique())

print("\nDuplicate row indices:")
display(
    review_df[review_df["index"].duplicated(keep=False)]
    [["index", "flag", "manual_classification"]]
    .sort_values("index")
)

Total review entries: 1180
Unique dataset rows: 1141

Duplicate row indices:


,index,flag,manual_classification
326,141,Unrelated Events,NaN
426,141,Foreign Location Mismatch,NaN
430,171,Foreign Location Mismatch,NaN
327,171,Unrelated Events,NaN
6,222,Healthcare,NaN
...,...,...,...
171,13879,Healthcare,NaN
194,15445,Healthcare,NaN
413,15445,Unrelated Events,NaN
416,17199,Unrelated Events,mismatch


Next step — Create a unique review dataset

In [28]:
unique_review_df = (
    review_df.sort_values("index")
    .groupby("index", as_index=False)
    .agg({
        "prompt": "first",
        "completion": "first",
        "flag": lambda x: ", ".join(sorted(set(x.dropna()))),
        "review_status": "first",
        "possible_categories": "first"
    })
)

unique_review_df["manual_classification"] = (
    unique_review_df["index"].map(review_labels)
)

print("Original entries:", len(review_df))
print("Unique records:", len(unique_review_df))
print("\nClassification counts:")
print(unique_review_df["manual_classification"].value_counts(dropna=False))

unique_review_df.to_csv(
    "/content/gramasetu_v2_unique_review.csv",
    index=False
)

print("\nUnique review file saved.")

Original entries: 1180
Unique records: 1141

Classification counts:
manual_classification
NaN             1095
mismatch          31
needs_review      11
relevant           4
Name: count, dtype: int64

Unique review file saved.


Next step — Prepare the remaining records for inspection

In [29]:
unclassified_df = unique_review_df[
    unique_review_df["manual_classification"].isna()
].copy()

unclassified_df.to_csv(
    "/content/gramasetu_v2_unclassified_candidates.csv",
    index=False
)

print("Unclassified unique records:", len(unclassified_df))
print("Saved successfully")

display(
    unclassified_df[
        ["index", "flag", "prompt", "completion"]
    ].head(10)
)

Unclassified unique records: 1095
Saved successfully


,index,flag,prompt,completion
0,0,Mathematics,Graph the region described by the system of inequalities\n\n$$\n\left\{\begin{array}{l}\nx+y-1 \leqslant 0 \quad(B C) \\\n-x+y-1 \leqslant 0 \quad(A B) \\\ny \geqslant-1 \quad(A C)\n\end{array}\right.\n$$,"To graph the region described by the system of inequalities, we determine the boundary lines, identify the vertices of the resulting shape, and define the shading for the region.\n\n### 1. Analyze the Boundary Lines\nWe rewrite each inequality in..."
1,17,Healthcare,Explain the key reasons why I should update my tax return.,Updating a previously filed tax return—formally done by filing Form 1040-X—is a strategic decision that ensures your financial record with the IRS is both accurate and optimized. There are several critical reasons to consider amending a return:\n...
2,18,Mathematics,"Formulate a hierarchical Bayesian model integrating spatially correlated gene‑expression covariance, phylogenetic branch‑length uncertainty, and non‑linear reaction‑diffusion dynamics to infer adaptive landscape curvature governing phenotypic tra...","This is a highly complex, multi-scale modeling problem that sits at the intersection of spatial statistics, phylogenetics, and evolutionary dynamics. Below is a formal formulation of a **Hierarchical Bayesian Spatio-Phylogenetic Reaction-Diffusio..."
3,50,Foreign Location Mismatch,Act as an agricultural expert specializing in pest management within the Kenyan context. Analyze the following user query and associated metadata to provide a comprehensive answer.\n\n**User Query:**\nAre there any specific plants that are more s...,"In the Nyeri region, certain plants are significantly more susceptible to mealybugs due to their physiology and growth habits.\n\n**Highly Susceptible Plants:**\n* **Coffee (Arabica):** The dominant cash crop here, highly vulnerable to the Coff..."
4,53,Mathematics,"Analyze the following system of differential equations to determine the values of the real constant \(a\) that result in resonance. Additionally, explain the general physical or mathematical principle illustrated by this phenomenon.\n\nSystem of ...","### 1. System Analysis\n\nTo analyze the dynamics of the system, we first express the given equations in a standard matrix form. The system is given by:\n\n\[\n\begin{aligned}\n\dot{x} &= -x + y + \cos 2t \\\n\dot{y} &= -5x + y + \cos 2t + 2a \si..."
5,58,Healthcare,"# Role\nYou are an expert analyst in African healthcare systems, specializing in the intersection of traditional medicine and modern biomedical practices.\n\n# Task\nAnalyze the provided text to answer the following core questions:\n1. How have t...","### Adaptation of Traditional African Healing Practices to Modern Healthcare Systems\n\nTraditional African healing practices have evolved from ancient indigenous systems—combining herbal medicine, spiritual beliefs, and communal rituals—to navig..."
6,60,Healthcare,"In addition to the given requirements, outline a robust access control mechanism that ensures patient privacy by restricting unauthorized access to medical records. Furthermore, propose an encryption scheme that secures the sensitive medical data...","This proposal outlines a comprehensive architecture for a Secure Electronic Health Record (EHR) System. It addresses access control, encryption, polyglot database schema design, disaster recovery, and dynamic data modeling, adhering to compliance..."
7,69,Foreign Location Mismatch,"Based on the following crop profile, suggest organic fertilizers that work well for this specific context:\n\n- **Crop**: Cabbage\n- **Location**: Kenya\n- **Date**: 2024-10\n- **Language**: English\n\nPlease provide recommendations tailored to t...","For cabbage cultivation in Kenya during the October short rains, use these organic fertilizers to support rapid leaf development, head firmness, and soil moisture retention:\n\n1. **Well-Decomposed Farmyard Manure (FYM):** Apply 4–5 tonnes per ac..."
8,75,Foreign Location Mi

Next step — Inspect a larger sample

In [30]:
sample_df = unclassified_df.sample(
    n=min(30, len(unclassified_df)),
    random_state=42
)

for _, row in sample_df.iterrows():
    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("FLAG:", row["flag"])
    print("PROMPT:", str(row["prompt"])[:700])
    print("COMPLETION:", str(row["completion"])[:700])


INDEX: 9279
FLAG: Foreign Location Mismatch
PROMPT: Provide a comprehensive guide on the vaccination schedule and requirements for chickens in Nyeri, Kenya, specifically for October 2024. Please ensure the response is in English and covers local disease risks, recommended vaccine types, administration
COMPLETION: **Poultry Vaccination Guide: Nyeri Region (October 2024)**

**Seasonal Context & Disease Risks**
October marks the onset of the short rains in Central Kenya. Increased humidity, temperature fluctuations, and rising mosquito populations elevate risks 

INDEX: 2202
FLAG: Healthcare
PROMPT: Analyze the current market conditions to identify and list specific bargains that validate the statement: "There are still plenty of bargains in the market." Provide examples across relevant sectors, explaining why each represents a strong value oppo
COMPLETION: While headline indices suggest a market stretched to its limits—driven largely by a narrow concentration of mega-cap technology name

Next step — Save these new classifications

In [31]:
new_review_labels = {
    9279: "needs_review",
    2202: "mismatch",
    1368: "needs_review",
    15038: "mismatch",
    17920: "needs_review",
    18856: "mismatch",
    4619: "mismatch",
    10471: "needs_review",
    19862: "relevant",
    12678: "mismatch",
    16865: "needs_review",
    17311: "needs_review",
    18069: "mismatch",
    10417: "needs_review",
    9870: "needs_review",
    11725: "relevant",
    9427: "relevant",
    9082: "needs_review",
    13807: "needs_review",
    13875: "needs_review",
    659: "needs_review",
    3511: "needs_review",
    2613: "mismatch",
    934: "needs_review",
    3375: "mismatch",
    12788: "needs_review",
    5016: "needs_review",
    6359: "mismatch",
    1465: "needs_review",
    1292: "needs_review"
}

review_labels.update(new_review_labels)

print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

Total manually labelled: 76
mismatch        40
needs_review    29
relevant         7
Name: count, dtype: int64


Next step: Review another 30 records

In [32]:
unclassified_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

print("Remaining unclassified:", len(unclassified_df))

sample_df = unclassified_df.sample(
    n=min(30, len(unclassified_df)),
    random_state=43
)

for _, row in sample_df.iterrows():
    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("FLAG:", row["flag"])
    print("PROMPT:", str(row["prompt"])[:700])
    print("COMPLETION:", str(row["completion"])[:700])

Remaining unclassified: 1065

INDEX: 18729
FLAG: Foreign Location Mismatch
PROMPT: Act as an agricultural expert specializing in sustainable farming practices in Kenya. Based on the specific context parameters provided below, identify and explain cover crops that are effective for pest control in potato fields.

**Context Parameter
COMPLETION: For potato fields in Kenya during the October 2024 short rains, the following cover crops are effective for pest management:

*   **White Mustard (*Brassica alba/juncea*)**
    *   **Target Pests:** Potato Cyst Nematodes (PCN) and root-knot nematodes

INDEX: 14178
FLAG: Foreign Location Mismatch
PROMPT: Analyze whether shade trees contribute to water conservation for coffee plants, specifically within the context of the following parameters:

- **Crop**: Coffee
- **Region**: Kenya
- **Timeframe**: 2024-09
- **Language**: English

Please provide a de
COMPLETION: Shade trees significantly contribute to water conservation for coffee crops in Kenya, 

Save these classifications

In [33]:
new_review_labels = {
    18729: "needs_review",
    14178: "needs_review",
    3200: "mismatch",
    302: "mismatch",
    3790: "mismatch",
    2428: "mismatch",
    5129: "mismatch",
    9182: "needs_review",
    5473: "mismatch",
    9289: "mismatch",
    4001: "needs_review",
    14516: "needs_review",
    18028: "mismatch",
    3298: "mismatch",
    15641: "needs_review",
    18843: "mismatch",
    14324: "needs_review",
    18898: "mismatch",
    11775: "needs_review",
    10006: "mismatch",
    13479: "mismatch",
    7903: "mismatch",
    17927: "mismatch",
    15256: "needs_review",
    13527: "needs_review",
    5829: "needs_review",
    3829: "mismatch",
    4254: "needs_review",
    7922: "needs_review",
    6718: "needs_review"
}

review_labels.update(new_review_labels)

print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

Total manually labelled: 106
mismatch        56
needs_review    43
relevant         7
Name: count, dtype: int64


Next step: Sample another 30

In [34]:
unclassified_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

print("Remaining unclassified:", len(unclassified_df))

sample_df = unclassified_df.sample(
    n=min(30, len(unclassified_df)),
    random_state=44
)

for _, row in sample_df.iterrows():
    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("FLAG:", row["flag"])
    print("PROMPT:", str(row["prompt"])[:700])
    print("COMPLETION:", str(row["completion"])[:700])

Remaining unclassified: 1035

INDEX: 4871
FLAG: Foreign Location Mismatch
PROMPT: <start_of_turn>user
Nk'omusomesa w'Abayizi abali mu Ggireedi 2 (Primary 2), baggwezeeko okugatta amaloboozi okusoma ebigambo ebyangu eby'engeri ya CVC (Consonant-Vowel-Consonant). Ng'atunuulidde obutereevu bw'Oluganda, kiki ekirina okuteekebwako enny
COMPLETION: Okuddamu: (A)

Lwaki: Oluganda lulina obutereevu mu ngeri gye lweewandiikamu (transparent orthography), ekitegeesa nti buli kigambo kisomesebwa mu bifaananyi by'amaloboozi ge. Nga abayizi ba Ggireedi 2 bamala okufuna obusobozi bw'okugatta amaloboozi 

INDEX: 12090
FLAG: Mathematics
PROMPT: Identify and explain three distinct effects that government spending or taxation can have on the economy.
COMPLETION: Fiscal policy—government spending and taxation—exerts profound influence over macroeconomic stability and the environment in which businesses operate. Three distinct effects of these interventions include the modulation of aggregate demand, the c

Save these five labels

In [35]:
new_review_labels = {
    4066: "needs_review",
    171: "needs_review",
    10452: "needs_review",
    15952: "needs_review",
    11380: "needs_review"
}

review_labels.update(new_review_labels)

print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

Total manually labelled: 111
mismatch        56
needs_review    48
relevant         7
Name: count, dtype: int64


Step 1: Retrieve the remaining records

In [36]:
sample_df = unclassified_df.sample(
    n=min(30, len(unclassified_df)),
    random_state=44
)

remaining_sample = sample_df[
    ~sample_df["index"].isin(review_labels.keys())
]

print("Remaining records in this batch:", len(remaining_sample))

for _, row in remaining_sample.iterrows():
    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("FLAG:", row["flag"])
    print("PROMPT:", str(row["prompt"])[:700])
    print("COMPLETION:", str(row["completion"])[:700])

Remaining records in this batch: 25

INDEX: 4871
FLAG: Foreign Location Mismatch
PROMPT: <start_of_turn>user
Nk'omusomesa w'Abayizi abali mu Ggireedi 2 (Primary 2), baggwezeeko okugatta amaloboozi okusoma ebigambo ebyangu eby'engeri ya CVC (Consonant-Vowel-Consonant). Ng'atunuulidde obutereevu bw'Oluganda, kiki ekirina okuteekebwako enny
COMPLETION: Okuddamu: (A)

Lwaki: Oluganda lulina obutereevu mu ngeri gye lweewandiikamu (transparent orthography), ekitegeesa nti buli kigambo kisomesebwa mu bifaananyi by'amaloboozi ge. Nga abayizi ba Ggireedi 2 bamala okufuna obusobozi bw'okugatta amaloboozi 

INDEX: 12090
FLAG: Mathematics
PROMPT: Identify and explain three distinct effects that government spending or taxation can have on the economy.
COMPLETION: Fiscal policy—government spending and taxation—exerts profound influence over macroeconomic stability and the environment in which businesses operate. Three distinct effects of these interventions include the modulation of aggregate demand

Save the classifications

In [37]:
new_review_labels = {
    4871: "mismatch",
    12090: "mismatch",
    6127: "mismatch",
    14159: "needs_review",
    2110: "needs_review",
    6655: "mismatch",
    3513: "needs_review",
    13038: "mismatch",
    13914: "needs_review",
    11089: "mismatch",
    4247: "needs_review",
    1949: "needs_review",
    10679: "needs_review",
    2548: "mismatch",
    391: "needs_review",
    963: "mismatch",
    4163: "mismatch",
    2406: "mismatch",
    914: "mismatch",
    10802: "needs_review",
    16366: "mismatch",
    18144: "mismatch",
    2177: "mismatch",
    15875: "mismatch",
    12431: "needs_review"
}

review_labels.update(new_review_labels)

print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

Total manually labelled: 136
mismatch        71
needs_review    58
relevant         7
Name: count, dtype: int64


Next step: Sample another 30 records

In [38]:
unclassified_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

print("Remaining unclassified:", len(unclassified_df))

sample_df = unclassified_df.sample(
    n=min(30, len(unclassified_df)),
    random_state=45
)

for _, row in sample_df.iterrows():
    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("FLAG:", row["flag"])
    print("PROMPT:", str(row["prompt"])[:700])
    print("COMPLETION:", str(row["completion"])[:700])

Remaining unclassified: 1005

INDEX: 18149
FLAG: Healthcare
PROMPT: What are the methods for tracking ransomware affiliate programs and actors?
COMPLETION: Tracking ransomware affiliate programs—specifically those operating under the Ransomware-as-a-Service (RaaS) model—is a complex discipline within Cyber Threat Intelligence (CTI). It requires distinguishing between the **Operators** (who develop the c

INDEX: 14294
FLAG: Healthcare
PROMPT: Analyze the following financial headline and provide a strategic guide for investors. Specifically, identify the key market indicators, economic data points, and risk factors an investor should monitor given the extreme divergence in forecasts descri
COMPLETION: ### Strategic Market Analysis: Navigating Extreme Forecast Dispersion

The widening dispersion in year-end S&P 500 forecasts—ranging from a 12% plunge to a 32% jump—signals a market at a critical inflection point. A 44% delta between the bearish and 

INDEX: 18981
FLAG: Foreign Location Mis

Save the classifications

In [39]:
new_review_labels = {
    18149: "mismatch",
    14294: "mismatch",
    18981: "needs_review",
    12646: "relevant",
    17475: "needs_review",
    18047: "needs_review",
    2396: "needs_review",
    7843: "needs_review",
    12205: "mismatch",
    4067: "mismatch",
    13558: "needs_review",
    12907: "needs_review",
    12927: "needs_review",
    16226: "mismatch",
    12611: "needs_review",
    8856: "mismatch",
    12613: "mismatch",
    1613: "needs_review",
    14602: "needs_review",
    13999: "mismatch",
    14369: "needs_review",
    463: "mismatch",
    18256: "mismatch",
    4655: "mismatch",
    6198: "mismatch",
    13884: "needs_review",
    17679: "needs_review",
    8587: "needs_review",
    15120: "needs_review",
    13041: "needs_review"
}

review_labels.update(new_review_labels)

print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

Total manually labelled: 166
mismatch        83
needs_review    75
relevant         8
Name: count, dtype: int64


Next step: Review the next 30 records

In [40]:
unclassified_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

print("Remaining unclassified:", len(unclassified_df))

sample_df = unclassified_df.sample(
    n=min(30, len(unclassified_df)),
    random_state=46
)

for _, row in sample_df.iterrows():
    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("FLAG:", row["flag"])
    print("PROMPT:", str(row["prompt"])[:700])
    print("COMPLETION:", str(row["completion"])[:700])

Remaining unclassified: 975

INDEX: 8664
FLAG: Foreign Location Mismatch
PROMPT: Act as an expert poultry farmer specializing in livestock management in Kenya. Provide a comprehensive guide on identifying signs of stress in hens and effective strategies to reduce it, specifically tailored to the context of Nyeri, Kenya, in Octobe
COMPLETION: ### 1. Identification of Stress in Hens

**Physical Indicators:**
*   **Comb and Wattles:** Pale, shriveled, or discolored appearance.
*   **Feathers:** Ruffled, dull, or broken feathers; bare patches from excessive pecking.
*   **Production:** Sudde

INDEX: 13433
FLAG: Foreign Location Mismatch
PROMPT: Please suggest companion plants that work well with the following drought-resistant fodder crops, taking into account the specific context provided below:

- Crop Type: generic
- Crop Type: generic
- Country: Kenya
- Region: Nakuru
- Date: 2024-10
- 
COMPLETION: For semi-arid conditions in Nakuru during the October short rains, the following compani

Update your labels

In [41]:
new_review_labels = {
    8664: "needs_review",
    13433: "needs_review",
    1343: "needs_review",
    4314: "needs_review",
    13229: "mismatch",
    13017: "mismatch",
    12971: "needs_review",
    17962: "needs_review",
    18148: "needs_review",
    17687: "needs_review",
    14179: "needs_review",
    18847: "mismatch",
    2012: "needs_review",
    2928: "mismatch",
    11536: "needs_review",
    18138: "needs_review",
    4994: "needs_review",
    19032: "mismatch",
    17719: "needs_review",
    12981: "mismatch",
    6686: "mismatch",
    505: "mismatch",
    16941: "mismatch",
    1104: "needs_review",
    543: "mismatch",
    19375: "needs_review",
    3448: "needs_review",
    5203: "mismatch",
    4642: "mismatch",
    6789: "needs_review"
}

review_labels.update(new_review_labels)

print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

Total manually labelled: 196
mismatch        95
needs_review    93
relevant         8
Name: count, dtype: int64


Next step: Sample the next 30 records

In [42]:
unclassified_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

print("Remaining unclassified:", len(unclassified_df))

sample_df = unclassified_df.sample(
    n=min(30, len(unclassified_df)),
    random_state=47
)

for _, row in sample_df.iterrows():
    print("\n" + "=" * 100)
    print("INDEX:", row["index"])
    print("FLAG:", row["flag"])
    print("PROMPT:", str(row["prompt"])[:700])
    print("COMPLETION:", str(row["completion"])[:700])

Remaining unclassified: 945

INDEX: 9176
FLAG: Healthcare
PROMPT: Role: Finance Expert

Objective: Provide a clear, concise, and informative response to the specific question below regarding poverty in the United States.

Instructions:
- Draw upon extensive financial expertise to address the core aspects of the que
COMPLETION: The United States presents a profound economic paradox: it possesses the largest nominal GDP and highest aggregate household wealth globally, yet maintains significant rates of poverty relative to its peer nations. Describing poverty in this context 

INDEX: 14837
FLAG: Healthcare
PROMPT: # Task
Generate a bulleted list of 5 key moments in Lyft's history based **strictly** on the provided text below.

# Constraints
- Extract exactly 5 distinct events.
- Ensure each bullet point is concise and factual.
- Do not include outside knowledg
COMPLETION: - Summer 2012: Logan Green and John Zimmer launched Lyft as a service of their existing long-distance carpooling compa

Update the classifications

In [43]:
new_review_labels = {
    9176: "mismatch",
    14837: "mismatch",
    18889: "needs_review",
    11777: "needs_review",
    11680: "mismatch",
    10608: "needs_review",
    17907: "needs_review",
    13672: "relevant",
    18920: "needs_review",
    11041: "mismatch",
    4668: "needs_review",
    69: "needs_review",
    10846: "needs_review",
    1139: "mismatch",
    10266: "needs_review",
    14045: "needs_review",
    18392: "mismatch",
    301: "needs_review",
    8178: "needs_review",
    12929: "needs_review",
    14490: "needs_review",
    480: "mismatch",
    9021: "needs_review",
    11698: "needs_review",
    7761: "mismatch",
    3232: "mismatch",
    15643: "mismatch",
    13543: "needs_review",
    19008: "needs_review",
    4756: "mismatch"
}

already_reviewed = {
    i: review_labels[i]
    for i in new_review_labels
    if i in review_labels
}

labels_to_add = {
    i: label
    for i, label in new_review_labels.items()
    if i not in review_labels
}

review_labels.update(labels_to_add)

print("Already classified:", already_reviewed)
print("New labels added:", len(labels_to_add))
print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

Already classified: {}
New labels added: 30
Total manually labelled: 226
needs_review    111
mismatch        106
relevant          9
Name: count, dtype: int64


Step 1 — Export the remaining records

In [44]:
from google.colab import files

remaining_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

export_df = remaining_df[["index", "flag", "prompt", "completion"]].copy()

export_df.to_csv(
    "/content/remaining_915_records.csv",
    index=False
)

print("Records exported:", len(export_df))

files.download("/content/remaining_915_records.csv")

Records exported: 915


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:
remaining_df[["index", "flag", "prompt", "completion"]].head(10)

,index,flag,prompt,completion
0,0,Mathematics,Graph the region described by the system of inequalities\n\n$$\n\left\{\begin{array}{l}\nx+y-1 \leqslant 0 \quad(B C) \\\n-x+y-1 \leqslant 0 \quad(A B) \\\ny \geqslant-1 \quad(A C)\n\end{array}\right.\n$$,"To graph the region described by the system of inequalities, we determine the boundary lines, identify the vertices of the resulting shape, and define the shading for the region.\n\n### 1. Analyze the Boundary Lines\nWe rewrite each inequality in..."
1,17,Healthcare,Explain the key reasons why I should update my tax return.,Updating a previously filed tax return—formally done by filing Form 1040-X—is a strategic decision that ensures your financial record with the IRS is both accurate and optimized. There are several critical reasons to consider amending a return:\n...
2,18,Mathematics,"Formulate a hierarchical Bayesian model integrating spatially correlated gene‑expression covariance, phylogenetic branch‑length uncertainty, and non‑linear reaction‑diffusion dynamics to infer adaptive landscape curvature governing phenotypic tra...","This is a highly complex, multi-scale modeling problem that sits at the intersection of spatial statistics, phylogenetics, and evolutionary dynamics. Below is a formal formulation of a **Hierarchical Bayesian Spatio-Phylogenetic Reaction-Diffusio..."
3,50,Foreign Location Mismatch,Act as an agricultural expert specializing in pest management within the Kenyan context. Analyze the following user query and associated metadata to provide a comprehensive answer.\n\n**User Query:**\nAre there any specific plants that are more s...,"In the Nyeri region, certain plants are significantly more susceptible to mealybugs due to their physiology and growth habits.\n\n**Highly Susceptible Plants:**\n* **Coffee (Arabica):** The dominant cash crop here, highly vulnerable to the Coff..."
4,53,Mathematics,"Analyze the following system of differential equations to determine the values of the real constant \(a\) that result in resonance. Additionally, explain the general physical or mathematical principle illustrated by this phenomenon.\n\nSystem of ...","### 1. System Analysis\n\nTo analyze the dynamics of the system, we first express the given equations in a standard matrix form. The system is given by:\n\n\[\n\begin{aligned}\n\dot{x} &= -x + y + \cos 2t \\\n\dot{y} &= -5x + y + \cos 2t + 2a \si..."
5,58,Healthcare,"# Role\nYou are an expert analyst in African healthcare systems, specializing in the intersection of traditional medicine and modern biomedical practices.\n\n# Task\nAnalyze the provided text to answer the following core questions:\n1. How have t...","### Adaptation of Traditional African Healing Practices to Modern Healthcare Systems\n\nTraditional African healing practices have evolved from ancient indigenous systems—combining herbal medicine, spiritual beliefs, and communal rituals—to navig..."
6,60,Healthcare,"In addition to the given requirements, outline a robust access control mechanism that ensures patient privacy by restricting unauthorized access to medical records. Furthermore, propose an encryption scheme that secures the sensitive medical data...","This proposal outlines a comprehensive architecture for a Secure Electronic Health Record (EHR) System. It addresses access control, encryption, polyglot database schema design, disaster recovery, and dynamic data modeling, adhering to compliance..."
8,75,Foreign Location Mismatch,"[{""role"": ""system"", ""content"": ""You are Hazina, a pan-African financial AI assistant developed by MsingiAI.\n\nYou are the CFO assistant for a M-Pesa agent based in Kenya.\nCurrency: KES (Kenyan Shilling)\nTax authority: KRA (Kenya Revenue Author...",The invoice has been successfully created and issued to Green Energy Solutions Kenya Ltd. Here are the key details:\n\n- **Invoice Number**: INV-2024-047 \n- **Issue Date**: 10 November 2024 \n- **Due Date**: 10 December 2024 \n- **Description...
9,81,Foreign Location 

In [46]:
new_review_labels = {
    0: "mismatch",
    1: "mismatch",
    2: "mismatch",
    3: "needs_review",
    4: "mismatch",
    5: "mismatch",
    6: "mismatch",
    8: "mismatch",
    9: "mismatch",
    10: "mismatch"
}

labels_to_add = {
    i: label
    for i, label in new_review_labels.items()
    if i not in review_labels
}

review_labels.update(labels_to_add)

print("New labels added:", len(labels_to_add))
print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

New labels added: 10
Total manually labelled: 236
mismatch        115
needs_review    112
relevant          9
Name: count, dtype: int64


In [47]:
remaining_df[["index", "flag", "prompt", "completion"]].head(20)

,index,flag,prompt,completion
0,0,Mathematics,Graph the region described by the system of inequalities\n\n$$\n\left\{\begin{array}{l}\nx+y-1 \leqslant 0 \quad(B C) \\\n-x+y-1 \leqslant 0 \quad(A B) \\\ny \geqslant-1 \quad(A C)\n\end{array}\right.\n$$,"To graph the region described by the system of inequalities, we determine the boundary lines, identify the vertices of the resulting shape, and define the shading for the region.\n\n### 1. Analyze the Boundary Lines\nWe rewrite each inequality in..."
1,17,Healthcare,Explain the key reasons why I should update my tax return.,Updating a previously filed tax return—formally done by filing Form 1040-X—is a strategic decision that ensures your financial record with the IRS is both accurate and optimized. There are several critical reasons to consider amending a return:\n...
2,18,Mathematics,"Formulate a hierarchical Bayesian model integrating spatially correlated gene‑expression covariance, phylogenetic branch‑length uncertainty, and non‑linear reaction‑diffusion dynamics to infer adaptive landscape curvature governing phenotypic tra...","This is a highly complex, multi-scale modeling problem that sits at the intersection of spatial statistics, phylogenetics, and evolutionary dynamics. Below is a formal formulation of a **Hierarchical Bayesian Spatio-Phylogenetic Reaction-Diffusio..."
3,50,Foreign Location Mismatch,Act as an agricultural expert specializing in pest management within the Kenyan context. Analyze the following user query and associated metadata to provide a comprehensive answer.\n\n**User Query:**\nAre there any specific plants that are more s...,"In the Nyeri region, certain plants are significantly more susceptible to mealybugs due to their physiology and growth habits.\n\n**Highly Susceptible Plants:**\n* **Coffee (Arabica):** The dominant cash crop here, highly vulnerable to the Coff..."
4,53,Mathematics,"Analyze the following system of differential equations to determine the values of the real constant \(a\) that result in resonance. Additionally, explain the general physical or mathematical principle illustrated by this phenomenon.\n\nSystem of ...","### 1. System Analysis\n\nTo analyze the dynamics of the system, we first express the given equations in a standard matrix form. The system is given by:\n\n\[\n\begin{aligned}\n\dot{x} &= -x + y + \cos 2t \\\n\dot{y} &= -5x + y + \cos 2t + 2a \si..."
5,58,Healthcare,"# Role\nYou are an expert analyst in African healthcare systems, specializing in the intersection of traditional medicine and modern biomedical practices.\n\n# Task\nAnalyze the provided text to answer the following core questions:\n1. How have t...","### Adaptation of Traditional African Healing Practices to Modern Healthcare Systems\n\nTraditional African healing practices have evolved from ancient indigenous systems—combining herbal medicine, spiritual beliefs, and communal rituals—to navig..."
6,60,Healthcare,"In addition to the given requirements, outline a robust access control mechanism that ensures patient privacy by restricting unauthorized access to medical records. Furthermore, propose an encryption scheme that secures the sensitive medical data...","This proposal outlines a comprehensive architecture for a Secure Electronic Health Record (EHR) System. It addresses access control, encryption, polyglot database schema design, disaster recovery, and dynamic data modeling, adhering to compliance..."
8,75,Foreign Location Mismatch,"[{""role"": ""system"", ""content"": ""You are Hazina, a pan-African financial AI assistant developed by MsingiAI.\n\nYou are the CFO assistant for a M-Pesa agent based in Kenya.\nCurrency: KES (Kenyan Shilling)\nTax authority: KRA (Kenya Revenue Author...",The invoice has been successfully created and issued to Green Energy Solutions Kenya Ltd. Here are the key details:\n\n- **Invoice Number**: INV-2024-047 \n- **Issue Date**: 10 November 2024 \n- **Due Date**: 10 December 2024 \n- **Description...
9,81,Foreign Location 

In [48]:
new_review_labels = {
    11: "mismatch",
    12: "needs_review",
    13: "mismatch",
    14: "needs_review",
    15: "needs_review",
    16: "needs_review",
    17: "mismatch",
    18: "mismatch",
    21: "mismatch",
    22: "needs_review"
}

labels_to_add = {
    i: label
    for i, label in new_review_labels.items()
    if i not in review_labels
}

review_labels.update(labels_to_add)

print("New labels added:", len(labels_to_add))
print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

New labels added: 10
Total manually labelled: 246
mismatch        120
needs_review    117
relevant          9
Name: count, dtype: int64


In [49]:
remaining_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

remaining_df[["index", "flag", "prompt", "completion"]].head(20)

,index,flag,prompt,completion
3,50,Foreign Location Mismatch,Act as an agricultural expert specializing in pest management within the Kenyan context. Analyze the following user query and associated metadata to provide a comprehensive answer.\n\n**User Query:**\nAre there any specific plants that are more s...,"In the Nyeri region, certain plants are significantly more susceptible to mealybugs due to their physiology and growth habits.\n\n**Highly Susceptible Plants:**\n* **Coffee (Arabica):** The dominant cash crop here, highly vulnerable to the Coff..."
4,53,Mathematics,"Analyze the following system of differential equations to determine the values of the real constant \(a\) that result in resonance. Additionally, explain the general physical or mathematical principle illustrated by this phenomenon.\n\nSystem of ...","### 1. System Analysis\n\nTo analyze the dynamics of the system, we first express the given equations in a standard matrix form. The system is given by:\n\n\[\n\begin{aligned}\n\dot{x} &= -x + y + \cos 2t \\\n\dot{y} &= -5x + y + \cos 2t + 2a \si..."
5,58,Healthcare,"# Role\nYou are an expert analyst in African healthcare systems, specializing in the intersection of traditional medicine and modern biomedical practices.\n\n# Task\nAnalyze the provided text to answer the following core questions:\n1. How have t...","### Adaptation of Traditional African Healing Practices to Modern Healthcare Systems\n\nTraditional African healing practices have evolved from ancient indigenous systems—combining herbal medicine, spiritual beliefs, and communal rituals—to navig..."
6,60,Healthcare,"In addition to the given requirements, outline a robust access control mechanism that ensures patient privacy by restricting unauthorized access to medical records. Furthermore, propose an encryption scheme that secures the sensitive medical data...","This proposal outlines a comprehensive architecture for a Secure Electronic Health Record (EHR) System. It addresses access control, encryption, polyglot database schema design, disaster recovery, and dynamic data modeling, adhering to compliance..."
8,75,Foreign Location Mismatch,"[{""role"": ""system"", ""content"": ""You are Hazina, a pan-African financial AI assistant developed by MsingiAI.\n\nYou are the CFO assistant for a M-Pesa agent based in Kenya.\nCurrency: KES (Kenyan Shilling)\nTax authority: KRA (Kenya Revenue Author...",The invoice has been successfully created and issued to Green Energy Solutions Kenya Ltd. Here are the key details:\n\n- **Invoice Number**: INV-2024-047 \n- **Issue Date**: 10 November 2024 \n- **Due Date**: 10 December 2024 \n- **Description...
9,81,Foreign Location Mismatch,"You are Hazina, a pan-African financial AI assistant developed by MsingiAI, serving as the CFO assistant for a microfinance institution in Uganda.\n\n**Operational Context & Constraints:**\n- **Currency:** UGX (Ugandan Shilling). Format all monet...","**Financial Crisis Analysis & Intervention Strategy**\n**Period:** October 2024\n**Status:** Critical – Immediate Action Required\n\nThe livestock disease outbreak has created a severe liquidity crisis, eliminating revenue while fixed operational..."
10,98,Healthcare,"Develop a strategic plan to help a small business navigate the complexities of healthcare regulations, focusing on minimizing financial risks and maximizing tax benefits. The plan must include at least three sections and adhere to the following s...",regulatory compliance and risk mitigation\n\nnavigating healthcare regulations requires a robust framework to avoid penalties. focus on hipaa privacy rules and stark law compliance to prevent fraud allegations. maintain detailed records of all pa...
11,118,Foreign Location Mismatch,"Generate a comprehensive report on poultry keepers in Kenya as of October 2024, written in English. The report should address the following key areas:\n\n1. **Industry Overview**: Current status, scale, and significance of poultry farming in Keny...","*

In [50]:
new_review_labels = {
    23: "needs_review",
    25: "needs_review",
    26: "needs_review"
}

labels_to_add = {
    i: label
    for i, label in new_review_labels.items()
    if i not in review_labels
}

review_labels.update(labels_to_add)

print("New labels added:", len(labels_to_add))
print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

New labels added: 3
Total manually labelled: 249
mismatch        120
needs_review    120
relevant          9
Name: count, dtype: int64


In [51]:
remaining_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

print("Remaining records:", len(remaining_df))
display(remaining_df[["index", "flag", "prompt", "completion"]].head(20))

Remaining records: 912


,index,flag,prompt,completion
3,50,Foreign Location Mismatch,Act as an agricultural expert specializing in pest management within the Kenyan context. Analyze the following user query and associated metadata to provide a comprehensive answer.\n\n**User Query:**\nAre there any specific plants that are more s...,"In the Nyeri region, certain plants are significantly more susceptible to mealybugs due to their physiology and growth habits.\n\n**Highly Susceptible Plants:**\n* **Coffee (Arabica):** The dominant cash crop here, highly vulnerable to the Coff..."
4,53,Mathematics,"Analyze the following system of differential equations to determine the values of the real constant \(a\) that result in resonance. Additionally, explain the general physical or mathematical principle illustrated by this phenomenon.\n\nSystem of ...","### 1. System Analysis\n\nTo analyze the dynamics of the system, we first express the given equations in a standard matrix form. The system is given by:\n\n\[\n\begin{aligned}\n\dot{x} &= -x + y + \cos 2t \\\n\dot{y} &= -5x + y + \cos 2t + 2a \si..."
5,58,Healthcare,"# Role\nYou are an expert analyst in African healthcare systems, specializing in the intersection of traditional medicine and modern biomedical practices.\n\n# Task\nAnalyze the provided text to answer the following core questions:\n1. How have t...","### Adaptation of Traditional African Healing Practices to Modern Healthcare Systems\n\nTraditional African healing practices have evolved from ancient indigenous systems—combining herbal medicine, spiritual beliefs, and communal rituals—to navig..."
6,60,Healthcare,"In addition to the given requirements, outline a robust access control mechanism that ensures patient privacy by restricting unauthorized access to medical records. Furthermore, propose an encryption scheme that secures the sensitive medical data...","This proposal outlines a comprehensive architecture for a Secure Electronic Health Record (EHR) System. It addresses access control, encryption, polyglot database schema design, disaster recovery, and dynamic data modeling, adhering to compliance..."
8,75,Foreign Location Mismatch,"[{""role"": ""system"", ""content"": ""You are Hazina, a pan-African financial AI assistant developed by MsingiAI.\n\nYou are the CFO assistant for a M-Pesa agent based in Kenya.\nCurrency: KES (Kenyan Shilling)\nTax authority: KRA (Kenya Revenue Author...",The invoice has been successfully created and issued to Green Energy Solutions Kenya Ltd. Here are the key details:\n\n- **Invoice Number**: INV-2024-047 \n- **Issue Date**: 10 November 2024 \n- **Due Date**: 10 December 2024 \n- **Description...
9,81,Foreign Location Mismatch,"You are Hazina, a pan-African financial AI assistant developed by MsingiAI, serving as the CFO assistant for a microfinance institution in Uganda.\n\n**Operational Context & Constraints:**\n- **Currency:** UGX (Ugandan Shilling). Format all monet...","**Financial Crisis Analysis & Intervention Strategy**\n**Period:** October 2024\n**Status:** Critical – Immediate Action Required\n\nThe livestock disease outbreak has created a severe liquidity crisis, eliminating revenue while fixed operational..."
10,98,Healthcare,"Develop a strategic plan to help a small business navigate the complexities of healthcare regulations, focusing on minimizing financial risks and maximizing tax benefits. The plan must include at least three sections and adhere to the following s...",regulatory compliance and risk mitigation\n\nnavigating healthcare regulations requires a robust framework to avoid penalties. focus on hipaa privacy rules and stark law compliance to prevent fraud allegations. maintain detailed records of all pa...
11,118,Foreign Location Mismatch,"Generate a comprehensive report on poultry keepers in Kenya as of October 2024, written in English. The report should address the following key areas:\n\n1. **Industry Overview**: Current status, scale, and significance of poultry farming in Keny...","*

In [52]:
old_review_labels = review_labels.copy()

review_labels = {
    int(unique_review_df.loc[i, "index"]): label
    for i, label in old_review_labels.items()
    if i in unique_review_df.index
}

remaining_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

print("Corrected classifications:", len(review_labels))
print("Remaining records:", len(remaining_df))
print(pd.Series(review_labels.values()).value_counts())

Corrected classifications: 41
Remaining records: 1100
mismatch        24
needs_review    16
relevant         1
Name: count, dtype: int64


In [53]:
display(remaining_df[["index", "flag", "prompt", "completion"]].head(20))

,index,flag,prompt,completion
7,69,Foreign Location Mismatch,"Based on the following crop profile, suggest organic fertilizers that work well for this specific context:\n\n- **Crop**: Cabbage\n- **Location**: Kenya\n- **Date**: 2024-10\n- **Language**: English\n\nPlease provide recommendations tailored to t...","For cabbage cultivation in Kenya during the October short rains, use these organic fertilizers to support rapid leaf development, head firmness, and soil moisture retention:\n\n1. **Well-Decomposed Farmyard Manure (FYM):** Apply 4–5 tonnes per ac..."
19,171,"Foreign Location Mismatch, Unrelated Events","You are Hazina, a pan-African financial AI assistant developed by MsingiAI, serving as the CFO assistant for a poultry farm in Uganda.\n\n**Operational Context:**\n- **Currency:** UGX (Ugandan Shilling)\n- **Tax Authority:** URA (Uganda Revenue A...","# Monthly Performance Review: October 2024 vs. September 2024\n\n## 1. Executive Summary\nThe poultry farm demonstrated robust growth in October 2024. Total income increased by **27.55%** month-over-month, while expenses grew at a slower rate of ..."
20,200,Unrelated Events,I am coordinating a bulk purchase initiative for our village cluster in Mahabubnagar to support 4 local Kirana stores and 2 Women's SHGs producing millet-based snacks. We need to aggregate demand for 3.5 tonnes of Grade-A Finger Millet (Ragi) for...,"For a volume of 29 tonnes (290 quintals) of Finger Millet (Ragi) in Telangana during September 2024, you have several viable channels to maximize returns. Given the bulk quantity, direct negotiations with processors and strategic mandi selection ..."
24,252,Foreign Location Mismatch,Bridging traditional knowledge and modern agronomy in Uganda taught me the value of clear communication; apply this to a Karnataka village scenario. The prompt involves farmers growing ragi and an SHG producing ragi malt needing to bulk-buy proce...,# Karnataka Ragi Procurement Plan: Bridging Tradition and Modern Logistics\n\nDrawing from the lesson learned in Uganda—where bridging traditional knowledge and modern agronomy required clear communication to prevent mistrust—this plan applies tr...
27,281,Mathematics,"# Henryson's Chauntecleer: Parodying Chivalric Romance\n\nRobert Henryson, the enigmatic Scottish makar of the late 15th century, is often celebrated for his *Moral Fables*, a collection of beast tales that, while drawing from Aesopic tradition, ...","This is a compelling and well-argued piece of literary criticism. You have successfully identified a specific lens—parody of chivalric romance—through which to view Henryson's *The Cock and the Jewel*, distinguishing it effectively from Chaucer's..."
28,301,Foreign Location Mismatch,"Agricultural evidence task — Kenya|Mandera County — unspecified — historical regional climate profile\n\nAct as a bilingual-ready agricultural evidence steward. Recover the requested result, show how it follows from the record, and prevent unsafe...","**1. Source-bound response**\nBased on the NASA POWER Climatology API (community AG) 20-year record, the typical historical climate and heat context for Mandera County, Kenya, in June is characterized by a mean air temperature of 25.19 °C, a dail..."
29,302,Foreign Location Mismatch,"[{""role"": ""system"", ""content"": ""You are Hazina, a pan-African financial AI assistant developed by MsingiAI.\n\nYou are the CFO assistant for a safari operator based in Tanzania.\nCurrency: TZS (Tanzanian Shilling)\nTax authority: TRA (Tanzania Re...","Based on the financial data for your fishing business in Dar es Salaam for Q1 2024 (January–March), here is a precise breakdown of your VAT liability, recent transactions, and M-Pesa reconciliation guidance. All figures comply with TRA regulation..."
30,314,Foreign Location Mismatch,"Reason through this ambiguous crop and region-specific field problem by combining growth stage, weather, and observations rather than matching one clue: Cassava in western Kenya is 

In [54]:
print("Total unique review records:", len(unique_review_df))
print("Total labels:", len(review_labels))
print("Unique original indices:", unique_review_df["index"].nunique())
print("Remaining records:", len(remaining_df))
print("Duplicate original indices:", unique_review_df["index"].duplicated().sum())
print("Labels matching dataset:", sum(i in set(unique_review_df["index"]) for i in review_labels))

Total unique review records: 1141
Total labels: 41
Unique original indices: 1141
Remaining records: 1100
Duplicate original indices: 0
Labels matching dataset: 41


In [55]:
display(remaining_df[["index", "flag", "prompt", "completion"]].head(20).to_string(index=False))

' index                                        flag                                                                                                                                                                                                                                                            prompt                                                                                                                                                                                                                                                     completion\n    69                   Foreign Location Mismatch Based on the following crop profile, suggest organic fertilizers that work well for this specific context:\\n\\n- **Crop**: Cabbage\\n- **Location**: Kenya\\n- **Date**: 2024-10\\n- **Language**: English\\n\\nPlease provide recommendations tailored to the growing    For cabbage cultivation in Kenya during the October short rains, use these organic fertilizers to support rapid leaf

In [56]:
new_review_labels = {
    69: "needs_review",
    171: "mismatch",
    200: "mismatch",
    252: "relevant",
    281: "mismatch",
    301: "mismatch",
    302: "mismatch",
    314: "needs_review",
    323: "mismatch",
    332: "needs_review",
    334: "needs_review",
    387: "needs_review",
    391: "needs_review",
    417: "mismatch",
    437: "mismatch",
    463: "mismatch",
    480: "mismatch",
    487: "mismatch",
    495: "needs_review",
    505: "mismatch"
}

labels_to_add = {
    i: label
    for i, label in new_review_labels.items()
    if i not in review_labels
}

review_labels.update(labels_to_add)

print("New labels added:", len(labels_to_add))
print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

New labels added: 20
Total manually labelled: 61
mismatch        36
needs_review    23
relevant         2
Name: count, dtype: int64


In [57]:
remaining_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

print("Remaining records:", len(remaining_df))
display(remaining_df[["index", "flag", "prompt", "completion"]].head(20).to_string(index=False))

Remaining records: 1080


' index                      flag                                                                                                                                                                                                                                                              prompt                                                                                                                                                                                                                                                        completion\n   542               Mathematics          Produce a comprehensive guide with detailed steps and tips on how to effectively create a personalized study schedule; including guidelines on organizing and prioritizing subjects, allocating study time, balancing studying with other activities, sett      # The Ultimate Guide to Creating a Personalized Study Schedule\\n\\nA well-structured study schedule is the bridge between where you are and where yo

In [58]:
new_review_labels = {
    542: "mismatch",
    543: "mismatch",
    580: "needs_review",
    659: "needs_review",
    671: "needs_review",
    680: "needs_review",
    682: "needs_review",
    685: "mismatch",
    691: "needs_review",
    703: "mismatch",
    723: "needs_review",
    733: "mismatch",
    736: "mismatch",
    737: "relevant",
    747: "needs_review",
    761: "mismatch",
    765: "needs_review",
    779: "needs_review",
    791: "needs_review",
    824: "mismatch"
}

labels_to_add = {
    i: label
    for i, label in new_review_labels.items()
    if i not in review_labels
}

review_labels.update(labels_to_add)

print("New labels added:", len(labels_to_add))
print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

remaining_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

print("\nRemaining records:", len(remaining_df))
display(
    remaining_df[["index", "flag", "prompt", "completion"]]
    .head(20)
    .to_string(index=False)
)

New labels added: 20
Total manually labelled: 81
mismatch        44
needs_review    34
relevant         3
Name: count, dtype: int64

Remaining records: 1060


' index                                   flag                                                                                                                                                                                                                                                               prompt                                                                                                                                                                                                                                                       completion\n   834              Foreign Location Mismatch        Generate a comprehensive report on the dairy industry in Nyeri, Kenya, for the period of October 2024. The report must be written in English.\\n\\nPlease address the following key areas:\\n- **Industry Overview**: Current state and significance of dairy fa    **Dairy Industry Report: Nyeri County, Kenya (October 2024)**\\n\\n**Industry Overview**\\nDairy farming remains a corner

In [59]:
new_review_labels = {
    834: "needs_review",
    836: "mismatch",
    841: "needs_review",
    860: "needs_review",
    878: "needs_review",
    879: "mismatch",
    912: "mismatch",
    914: "mismatch",
    923: "mismatch",
    934: "needs_review",
    963: "mismatch",
    981: "mismatch",
    1012: "mismatch",
    1027: "needs_review",
    1053: "mismatch",
    1069: "needs_review",
    1072: "needs_review",
    1084: "needs_review",
    1104: "needs_review",
    1126: "relevant"
}

labels_to_add = {
    i: label
    for i, label in new_review_labels.items()
    if i not in review_labels
}

review_labels.update(labels_to_add)

print("New labels added:", len(labels_to_add))
print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

remaining_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

print("\nRemaining records:", len(remaining_df))
display(
    remaining_df[["index", "flag", "prompt", "completion"]]
    .head(20)
    .to_string(index=False)
)

New labels added: 20
Total manually labelled: 101
mismatch        53
needs_review    44
relevant         4
Name: count, dtype: int64

Remaining records: 1040


' index                                        flag                                                                                                                                                                                                                                                           prompt                                                                                                                                                                                                                                                         completion\n  1139                                  Healthcare       Draft a compliant healthcare job advertisement for Liberty Senior Living targeting a Registered Nurse (RN) for a Full-Time Days position. Begin with the branding header "Liberty Cares With Compassion". Immediately follow with a highlighted incentive  Liberty Cares With Compassion\\n\\n***$10,000 Sign on Bonus***\\n\\n**Position:** Registered Nurse (RN) – Full-Time Days\\n

In [60]:
new_review_labels = {
    1139: "mismatch",
    1149: "needs_review",
    1186: "needs_review",
    1209: "mismatch",
    1235: "mismatch",
    1250: "mismatch",
    1292: "needs_review",
    1343: "needs_review",
    1368: "needs_review",
    1385: "relevant",
    1396: "mismatch",
    1397: "needs_review",
    1409: "mismatch",
    1410: "mismatch",
    1412: "mismatch",
    1453: "mismatch",
    1465: "mismatch",
    1487: "mismatch",
    1488: "mismatch",
    1489: "needs_review"
}

labels_to_add = {
    i: label
    for i, label in new_review_labels.items()
    if i not in review_labels
}

review_labels.update(labels_to_add)

print("New labels added:", len(labels_to_add))
print("Total manually labelled:", len(review_labels))
print(pd.Series(review_labels.values()).value_counts())

remaining_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

print("\nRemaining records:", len(remaining_df))
display(
    remaining_df[["index", "flag", "prompt", "completion"]]
    .head(20)
    .to_string(index=False)
)

New labels added: 20
Total manually labelled: 121
mismatch        65
needs_review    51
relevant         5
Name: count, dtype: int64

Remaining records: 1020


' index                      flag                                                                                                                                                                                                                                                           prompt                                                                                                                                                                                                                                                         completion\n  1495 Foreign Location Mismatch     Agricultural evidence task — UG|Busoga Farming System — Chilli, dry — regional crop-calendar extraction\\n\\nConvert the structured agricultural record into a practical answer without filling gaps from general knowledge or turning historical evidence int    1. Decision\\nAccording to the FAO Crop Calendar, the usual sowing window for Chilli, dry in Busoga Farming System, Uganda is 25 February to 30 September,

In [61]:
print("Total flagged:", len(unique_review_df))
print("Classified:", len(review_labels))
print("Remaining:", len(unique_review_df) - len(review_labels))


Total flagged: 1141
Classified: 121
Remaining: 1020


In [62]:
remaining_df = unique_review_df[
    ~unique_review_df["index"].isin(review_labels.keys())
].copy()

batch_df = remaining_df.head(100)

for _, row in batch_df.iterrows():
    print(f"\nINDEX: {row['index']} | FLAG: {row['flag']}")
    print("PROMPT:", str(row["prompt"])[:500])
    print("COMPLETION:", str(row["completion"])[:500])
    print("-" * 80)


INDEX: 1495 | FLAG: Foreign Location Mismatch
PROMPT: Agricultural evidence task — UG|Busoga Farming System — Chilli, dry — regional crop-calendar extraction

Convert the structured agricultural record into a practical answer without filling gaps from general knowledge or turning historical evidence int
COMPLETION: 1. Decision
According to the FAO Crop Calendar, the usual sowing window for Chilli, dry in Busoga Farming System, Uganda is 25 February to 30 September, and the harvest window is 10 May to 30 January.

2. Source trace
Record: FAO Crop Calendar API.
L
--------------------------------------------------------------------------------

INDEX: 1508 | FLAG: Foreign Location Mismatch
PROMPT: Act as an expert livestock veterinarian familiar with cattle farming in Kenya. Based on the context provided below, list and explain the common signs of heat (estrus) in a cow.

Context:
- Topic: livestock
- Animal: Cattle
- Location: Kenya, Nyeri
- 
COMPLETION: To identify heat (estrus) in you